# 0.0 Input Tables

In [0]:
# %sql
# CREATE OR REPLACE TEMPORARY VIEW prod_sales_list as
# select account_owner_amperity_id as amperity_id,  prods.* 
# from mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_onstar_account account
# left join mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_onstar_vehicle_subscription sub using (account_nbr)
# left join mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_onstar_product_subscriptions prods using(subscription_nkey)

# WHERE 
#   account_country_cd = 'US'
#   and prods.price_plan_cd IN ('MTHLY','SUBSCBRPPD','SBSCBRPPD','COMP')
#   and prods.prod_type_cd = 'CORE'
#   and prods.prod_create_dt >= '2024-06-05'



# select distinct accnt.account_owner_amperity_id, accnt.account_nbr, sub.vin, 
# account_is_open_ind, account_status_cd, account_status_nm, account_owner_status_cd,
# vehicle_status_cd,
# unit.unit_gen_num,  
# unit.unit_state_desc
# from mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_onstar_account as accnt
# left join mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_onstar_vehicle_subscription sub using (account_nbr)
# left join mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_onstar_product_subscriptions prods using(subscription_nkey)
# left join mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_vehicle vin using (vin)
# left join mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_onstar_unit unit on vin.onstar_station_id = unit.station_id
# where account_owner_amperity_id in ("c56b1617-f596-31ab-b006-6b9121696af0") --("4ce214fa-c0d7-3f5b-817d-5cc25a71d448")


In [0]:
%sql
-- No permissino to access the schema
--select * from customer_recognition.crosswalk_public.crosswalk_id_graph limit 1;

In [0]:
%sql


--select * from acquire.gold_connected_vehicle.member_base_history limit 1;

-- select distinct amperity_id, acct.vin, account_number, is_current_snapshot, model, brand, model_year  
--   from acquire.gold_connected_vehicle.member_base_history as acct
--   left join acquire.gold_connected_vehicle.member_base_vehicle_dim as vdims on acct.vin = vdims.vin
--   where 
--   amperity_id in ("00290603-1e2e-3823-ba70-67bc7993af2d")
--   --("c56b1617-f596-31ab-b006-6b9121696af0") 
--   --( "4ce214fa-c0d7-3f5b-817d-5cc25a71d448") --"0002f396-e5f1-3281-91aa-e75dd42d0a43",
--   --and is_current_snapshot = true; 
--   --VIN in("3GNKBCRS2LS707579")



In [0]:
%sql

-- About source of data coming from

-- select vin.vin
-- from customer.silver_individual.individual_amperity_xref_src_key amp
-- join acquire.vehicle_silver.customer_vin_relation vin
--     on amp.source_pk1 = vin.individual_business_id
-- where amp.src_sys_cd = 'CMDS_146492'
-- limit 10;

--select * from customer.silver_individual.individual_amperity_xref_src_key limit 1;
--select * from acquire.vehicle_silver.customer_vin_relation limit 1;


In [0]:
%sql


-- SELECT DISTINCT amperity_id, ids.account_nbr, vdims.vin, revised_model_nm, make_nm, vin_model_year_nbr
-- FROM  acquire.silver_connected_vehicle.connected_services_customer as ids
--   LEFT join acquire.silver_connected_vehicle.account_vehicle_subscription as acct
--   on acct.account_nbr = ids.account_nbr

--   LEFT JOIN acquire.silver_connected_vehicle.digital_vehicle as vdims
--   on vdims.vin = acct.vin
-- where 
-- -- acct.account_nbr in (182351677)
-- amperity_id in ("00aaad37-0be8-32af-bedb-2ad6617148c7")
-- --("001537be-3bee-31ac-ba33-fd0021e3a145")
-- --("c56b1617-f596-31ab-b006-6b9121696af0")
-- --("4ce214fa-c0d7-3f5b-817d-5cc25a71d448") 
-- --vdims.VIN in ("3GNKBCRS2LS707579")



# 0.0 Import Library

In [0]:
########### Move Import Library to Widgets Section

# import pandas as pd
# import statsmodels.api as sm
# from scipy.stats import chi2_contingency
# from datetime import datetime, timedelta


# # Set display options
# pd.set_option('display.max_rows', None)  # Show all rows
# pd.set_option('display.max_columns', None)  # Show all columns
# pd.set_option('display.width', 1000)  # Set the width of the display
# pd.set_option('display.colheader_justify', 'center')  # Align column headers

# pd.options.display.float_format = '{:,.4f}'.format 

# 0.0 Widgets

In [0]:
import pandas as pd
import statsmodels.api as sm
from scipy.stats import chi2_contingency
from datetime import datetime, timedelta


# Set display options
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', 1000)  # Set the width of the display
pd.set_option('display.colheader_justify', 'center')  # Align column headers

pd.options.display.float_format = '{:,.4f}'.format 


# Create a text widget for date input
dbutils.widgets.text("First_Ever_date", "2024-01-01", "Enter List Pull Date Ever [00]") ## The very first LP dates, used to identify FLP for each Amp_ID
dbutils.widgets.text("FLP_date", "2024-01-01", "Enter List Pull Date [First]")
dbutils.widgets.text("Max_lp_date", "2024-01-01", "Enter List Pull Date [Last]")
dbutils.widgets.text("Gap_Days", "0", "Enter X days after FLP Date")

dbutils.widgets.text("conv_start_range_date", "2024-01-01", "Enter Conversion Date [First]")
dbutils.widgets.text("conv_end_date", "", "Enter Conversion Date [Last]")

# Retrieve the values entered by the user
FLP_date = dbutils.widgets.get("FLP_date")
FLP_ever_date = dbutils.widgets.get("First_Ever_date")
Max_lp_date = dbutils.widgets.get("Max_lp_date")
X_days = int(dbutils.widgets.get("Gap_Days"))


conv_start_range_date = dbutils.widgets.get("conv_start_range_date")
conv_start_range_dt = datetime.strptime(conv_start_range_date, "%Y-%m-%d")

conv_end_date = dbutils.widgets.get("conv_end_date")

# Convert FLP_date to datetime object
FLP_date_dt = datetime.strptime(FLP_date, "%Y-%m-%d")
Max_lp_date_dt = datetime.strptime(Max_lp_date, "%Y-%m-%d")
FLP_ever_date_dt = datetime.strptime(FLP_ever_date, "%Y-%m-%d")

if conv_end_date == "":
  conv_end_date_dt = datetime.today().strftime('%Y-%m-%d')
  conv_end_date = datetime.today().strftime('%Y-%m-%d')
else:
  conv_end_date_dt = datetime.strptime(conv_end_date, "%Y-%m-%d")
  

# Calculate target date
#conv_start_range_date = conv_start_range_dt + timedelta(days=0)
conv_start_range_date = datetime.strptime(conv_start_range_date, "%Y-%m-%d")
conv_start_date = FLP_date_dt + timedelta(days=X_days)


print(f"First List Pull Date: {FLP_date} | dt {FLP_date_dt}")
print(f"Max List Pull Date: {Max_lp_date} | dt {Max_lp_date_dt}")
print(f"Conversion Start Date: {conv_start_date}")
print(f"Conversion Start Range Date: {conv_start_range_date}")
print(f"Conversion End Date: {conv_end_date} | dt {conv_end_date_dt}")

#print(f"Conversion Start Date: {conv_start_date.strftime('%Y-%m-%d')}")
#print(f"Conversion Start Ragne Date: {conv_start_range_date.strftime('%Y-%m-%d')}")
#print(f"Conversion End Date: {conv_end_date.strftime('%Y-%m-%d')}")




First List Pull Date: 2024-10-28 | dt 2024-10-28 00:00:00
Max List Pull Date: 2025-01-20 | dt 2025-01-20 00:00:00
Conversion Start Date: 2024-10-28 00:00:00
Conversion Start Range Date: 2025-01-01 00:00:00
Conversion End Date: 2025-02-10 | dt 2025-02-10 00:00:00


In [0]:
# Create a multiselect widget for campaign names
dbutils.widgets.multiselect("campaign_nm_list", "OnStar_Trialist", 
                            [
                            ######### Connected ###############
                            "OnStar_Connected", "OnStar_Connected_Paid", "OnStar_Connected_Paid_Media", 
                             "OnStar_Dormant", "OnStar_Trialist", "OnStar_Winback",
                             "OnStar_Credit_Card_Decline","OnStar_Guardian","OnStar_SuperCruise",
                             
                             # Remote Start
                             "OnStar_Buick_RemoteStart-RAPlan-LoggedInL90Days","OnStar_Buick_RemoteStart-RAPlan-NoLoggedInL90Days",
                             "OnStar_Cadillac_RemoteStart-RAPlan-LoggedInL90Days","OnStar_Cadillac_RemoteStart-RAPlan-NoLoggedInL90Days",
                             "OnStar_Chevrolet_RemoteStart-RAPlan-LoggedInL90Days","OnStar_Chevrolet_RemoteStart-RAPlan-NoLoggedInL90Days",
                             "OnStar_GMC_RemoteStart-RAPlan-LoggedInL90Days","OnStar_GMC_RemoteStart-RAPlan-NoLoggedInL90Days",

                             ################# myBrand ###########################
                             ############### NonRA Plan ##################
                             "OnStar_Buick_myBrand-NonRAPlan-or-RARenewal90Days",
                             "OnStar_Cadillac_myBrand-NonRAPlan-or-RARenewal90Days",
                             "OnStar_Chevrolet_myBrand-NonRAPlan-or-RARenewal90Days",
                             "OnStar_GMC_myBrand-NonRAPlan-or-RARenewal90Days",

                             ############### RA Plan ##################
                             "OnStar_Buick_myBrand-RAPlan-LoggedInL90Days","OnStar_Buick_myBrand-RAPlan-NoLoggedInL90Days",
                             "OnStar_Cadillac_myBrand-RAPlan-LoggedInL90Days","OnStar_Cadillac_myBrand-RAPlan-NoLoggedInL90Days","OnStar_Chevrolet_myBrand-RAPlan-LoggedInL90Days","OnStar_Chevrolet_myBrand-RAPlan-NoLoggedInL90Days",
                             "OnStar_GMC_myBrand-RAPlan-LoggedInL90Days","OnStar_GMC_myBrand-RAPlan-NoLoggedInL90Days",

                             ############### DRPO ##################
                             "OnStar_DRPO-3Year-RAPlan-NoPaidCorePlan"
                             ], 
                            "Enter or Select Campaign Names")

campaign_nm_list = dbutils.widgets.get("campaign_nm_list").split(",")  # List of selected names
campaign_nm = ", ".join([f"'{name.strip()}'" for name in campaign_nm_list])  # Format for SQL

print(f"Selected campaign names for SQL: {campaign_nm}")

Selected campaign names for SQL: 'OnStar_Connected', 'OnStar_Connected_Paid', 'OnStar_Connected_Paid_Media'


In [0]:
dbutils.widgets.dropdown("fix_campaign", "N", ["Y", "N"], "Fix Campaign Name?")

fix_campaign = dbutils.widgets.get("fix_campaign")

In [0]:
dbutils.widgets.dropdown("lift_conv_type", "Multiple Orders", ["First Order Only", "Multiple Orders"], "Lift Conversion Type")

df_conv_input = dbutils.widgets.get("lift_conv_type")

#df_conv_base = df_order_first


In [0]:
dbutils.widgets.dropdown("LP_Freq", "Daily", ["Daily", "Weekly"], "List Pull Frequency")

LP_Freq = dbutils.widgets.get("LP_Freq")


In [0]:

# Build a Python list of date strings
current = FLP_ever_date_dt
date_list = []
while current <= Max_lp_date_dt:
    date_list.append(current.strftime("%Y-%m-%d"))
    current += timedelta(days=1)

# 3. Create a dropdown widget for the user to pick one date
#    Remove the existing widget if it’s already defined
try:
    dbutils.widgets.remove("weekly_date")
except:
    pass

# If date_list is empty or invalid, handle gracefully
if not date_list:
    date_list = ["No valid dates found"]

default_value = date_list[0]  # first date as a default
# Create the dropdown widget with dynamic choices
dbutils.widgets.multiselect(
    name        = "weekly_date",
    defaultValue= default_value,
    choices     = date_list,
    label       = "Select a Date"
)


## Read in weekly_date
selected_dates_str = dbutils.widgets.get("weekly_date")  

## Split on commas to get a Python list
selected_dates_list = [d.strip() for d in selected_dates_str.split(",") if d.strip()]

# 3) Build the parenthesized string for SQL: ('2024-10-23','2024-10-24')
weekly_dates = "(" + ",".join(f"'{date}'" for date in selected_dates_list) + ")"
print(f"Selected dates: {weekly_dates}")

Selected dates: ('2024-10-23','2024-10-24','2024-10-28','2024-11-04','2024-11-11','2024-11-18','2025-01-17')


In [0]:
dbutils.widgets.dropdown("pull_camp_aud", "N", ["Y", "N"], "0 Run AEP Audience Size?")

pull_camp_aud = dbutils.widgets.get("pull_camp_aud")

In [0]:
dbutils.widgets.dropdown("check_multi_status", "N", ["Y", "N"], "0 Run AEP Multi status?")

check_multi_status = dbutils.widgets.get("check_multi_status")

In [0]:
# Remove a specific widget
#dbutils.widgets.remove("weekly_date")

# Remove all widgets
#dbutils.widgets.removeAll()

In [0]:
%sql
--select * from     mktg_dmp_prod.a146492_cmds_bronze.aep_campaign_audience_hist limit 10

# 1. AEP Audience Files

In [0]:
##### Proccess, add extra columns and create temp veiw for aep_campiagn_audience_hist table
spark.sql(f"""
          
SELECT *, to_date(aep_export_ts) AS lp_date,
    MIN(TO_DATE(aep_export_ts)) OVER (PARTITION BY amperity_id, audience_nm) AS first_lp_date,
    MAX(TO_DATE(aep_export_ts)) OVER (PARTITION BY amperity_id, audience_nm) AS last_lp_date,
    row_number() OVER (PARTITION BY amperity_id, audience_nm ORDER BY TO_DATE(aep_export_ts)) AS row_nm_flp,
    LEFT(audience_nm, LENGTH(audience_nm) - INSTR(REVERSE(audience_nm), '_')) AS campaign_name,
    RIGHT(audience_nm, INSTR(REVERSE(audience_nm), '_') - 1) AS exposure
    FROM 
    --mktg_dmp_uat.a146492_cmds_bronze.aep_campaign_audience_hist
    mktg_dmp_prod.a146492_cmds_bronze.aep_campaign_audience_hist --New table since 10/24

""").createOrReplaceTempView("AEP_camp_aud_hist_zz_v")



##### Create View AEP_camp_aud_v before Max_lp_date only which will be used to match orders

##############################################################################################


if LP_Freq == "Weekly":
    print (f"{LP_Freq} Selected, weeks: {weekly_dates}")
    where_clause = f"to_date(aep_export_ts) in {weekly_dates}"
elif LP_Freq == "Daily":
    print (f"{LP_Freq} Selected, Days condition: <{Max_lp_date}")
    where_clause = f"to_date(aep_export_ts) <= '{Max_lp_date}'"
else:
    print (f"{LP_Freq} Selected, Days condition: <{Max_lp_date}")
    where_clause = f"to_date(aep_export_ts) <= '{Max_lp_date}'"

##############################################################################################

spark.sql(f"""
          
-- %sql
-- DROP TEMPORARY VARIABLE IF EXISTS min_date;
-- DROP TEMPORARY VARIABLE IF EXISTS max_date;
-- DECLARE VARIABLE min_date STRING; 
-- DECLARE VARIABLE max_date STRING;
-- SET VARIABLE min_date = '2024-10-10';
-- SET VARIABLE max_date = '2024-11-10'; -- MAX List Pull Date

--DROP VIEW IF EXISTS AEP_camp_aud_v; CREATE temp VIEW AEP_camp_aud_v AS()


SELECT *, to_date(aep_export_ts) AS lp_date,
    MIN(TO_DATE(aep_export_ts)) OVER (PARTITION BY amperity_id, audience_nm) AS first_lp_date,
    MAX(TO_DATE(aep_export_ts)) OVER (PARTITION BY amperity_id, audience_nm) AS last_lp_date,
    row_number() OVER (PARTITION BY amperity_id, audience_nm ORDER BY TO_DATE(aep_export_ts)) AS row_nm_flp,
    LEFT(audience_nm, LENGTH(audience_nm) - INSTR(REVERSE(audience_nm), '_')) AS campaign_name,
    RIGHT(audience_nm, INSTR(REVERSE(audience_nm), '_') - 1) AS exposure
    FROM 
    --mktg_dmp_uat.a146492_cmds_bronze.aep_campaign_audience_hist
    mktg_dmp_prod.a146492_cmds_bronze.aep_campaign_audience_hist --New table since 10/24
    WHERE 
    {where_clause}
    
      --to_date(aep_export_ts) >= min_date
      --to_date(aep_export_ts) BETWEEN min_date AND max_date
      --to_date(aep_export_ts) <= '{Max_lp_date}'
      --to_date(aep_export_ts) in ('2024-10-23','2024-10-28','2024-11-04','2024-11-11','2024-11-18')
      --to_date(aep_export_ts) in ('2024-10-23','2024-10-24','2024-10-28','2024-11-04','2024-11-11','2024-11-18','2024-11-25')
      --to_date(aep_export_ts) in ('2024-11-04','2024-11-11','2024-11-18','2024-11-25','2024-11-27','2024-12-02','2024-12-09','2024-12-16','2024-12-23','2024-12-30')

--select * from AEP_camp_aud_v order by date ASC limit 10;

-- select distinct campaign_name, exposure, audience_nm, min(lp_date) as min_lp_date, max(lp_date) as max_lp_date
--   from AEP_camp_aud_v group by All
--   order by campaign_name, exposure;

--SELECT amperity_id, audience_nm, lp_date, first_lp_date FROM AEP_camp_aud_v order by amperity_id, audience_nm, lp_date limit 100;

""").createOrReplaceTempView("AEP_camp_aud_v")

print(f"Distinct Weeks:\n")
dt_lp_dates = spark.sql(f"""
          select distinct first_lp_date from AEP_camp_aud_v 
          where campaign_name in ({campaign_nm})
          order by first_lp_date ASC;
""").toPandas()
display(dt_lp_dates)

last_listpull_date = dt_lp_dates.iloc[-1,0]
print(f"Last List Pull Date: {last_listpull_date}\n")



print(f"AEP_camp_aud_v Sample Data:\n")
spark.sql(f"""
          select * from AEP_camp_aud_v 
          where campaign_name in ({campaign_nm})
          order by audience_nm, amperity_id, lp_date ASC limit 10;
""").display()


Weekly Selected, weeks: ('2024-10-23','2024-10-24','2024-10-28','2024-11-04','2024-11-11','2024-11-18','2025-01-17')
Distinct Weeks:



first_lp_date
2024-10-23
2024-10-24
2024-10-28
2024-11-04
2024-11-11
2024-11-18
2025-01-17


Last List Pull Date: 2025-01-17

AEP_camp_aud_v Sample Data:



audience_nm,amperity_id,aep_export_ts,audience_filename,inserted_timestamp,lp_date,first_lp_date,last_lp_date,row_nm_flp,campaign_name,exposure
OnStar_Connected_Holdout,000001d8-8ba9-3c06-a005-7c71340f9ec2,2024-10-23T18:21:00Z,AEP_US_OnStar_Connected_Holdout_20241023_182100.json.gz,2024-10-24T03:07:15.889Z,2024-10-23,2024-10-23,2025-01-17,1,OnStar_Connected,Holdout
OnStar_Connected_Holdout,000001d8-8ba9-3c06-a005-7c71340f9ec2,2024-10-24T18:12:28Z,AEP_US_OnStar_Connected_Holdout_20241024_181228.json.gz,2024-10-25T03:05:46.603Z,2024-10-24,2024-10-23,2025-01-17,2,OnStar_Connected,Holdout
OnStar_Connected_Holdout,000001d8-8ba9-3c06-a005-7c71340f9ec2,2024-10-28T18:24:15Z,AEP_US_OnStar_Connected_Holdout_20241028_182415.json.gz,2024-10-29T03:06:04.578Z,2024-10-28,2024-10-23,2025-01-17,3,OnStar_Connected,Holdout
OnStar_Connected_Holdout,000001d8-8ba9-3c06-a005-7c71340f9ec2,2024-11-04T18:14:57Z,AEP_US_OnStar_Connected_Holdout_20241104_181457.json.gz,2024-11-05T04:05:43.401Z,2024-11-04,2024-10-23,2025-01-17,4,OnStar_Connected,Holdout
OnStar_Connected_Holdout,000001d8-8ba9-3c06-a005-7c71340f9ec2,2024-11-11T18:05:18Z,AEP_US_OnStar_Connected_Holdout_20241111_180518.json.gz,2024-11-12T04:05:28.675Z,2024-11-11,2024-10-23,2025-01-17,5,OnStar_Connected,Holdout
OnStar_Connected_Holdout,000001d8-8ba9-3c06-a005-7c71340f9ec2,2024-11-18T18:13:25Z,AEP_US_OnStar_Connected_Holdout_20241118_181325.json.gz,2024-11-19T04:06:53.596Z,2024-11-18,2024-10-23,2025-01-17,6,OnStar_Connected,Holdout
OnStar_Connected_Holdout,000001d8-8ba9-3c06-a005-7c71340f9ec2,2025-01-17T18:08:49Z,AEP_US_OnStar_Connected_Holdout_20250117_180849.json.gz,2025-01-18T04:07:13.779Z,2025-01-17,2024-10-23,2025-01-17,7,OnStar_Connected,Holdout
OnStar_Connected_Holdout,000011b6-fe5b-39dd-9dc5-ede9d2e912aa,2024-10-23T18:21:00Z,AEP_US_OnStar_Connected_Holdout_20241023_182100.json.gz,2024-10-24T03:07:15.889Z,2024-10-23,2024-10-23,2025-01-17,1,OnStar_Connected,Holdout
OnStar_Connected_Holdout,000011b6-fe5b-39dd-9dc5-ede9d2e912aa,2024-10-24T18:12:28Z,AEP_US_OnStar_Connected_Holdout_20241024_181228.json.gz,2024-10-25T03:05:46.603Z,2024-10-24,2024-10-23,2025-01-17,2,OnStar_Connected,Holdout
OnStar_Connected_Holdout,000011b6-fe5b-39dd-9dc5-ede9d2e912aa,2024-10-28T18:24:15Z,AEP_US_OnStar_Connected_Holdout_20241028_182415.json.gz,2024-10-29T03:06:04.578Z,2024-10-28,2024-10-23,2025-01-17,3,OnStar_Connected,Holdout


In [0]:
%sql
--select * from mktg_dmp_prod.a146492_cmds_bronze.aep_campaign_audience_hist limit 10

In [0]:
if pull_camp_aud == "Y": 

  spark.sql(f"""
            select audience_nm, campaign_name, exposure, first_lp_date, count(distinct amperity_id) as cnt_amp
            FROM AEP_camp_aud_hist_zz_v
            where row_nm_flp = 1
            group by all
            order by audience_nm, campaign_name, exposure, first_lp_date;
  """).display()

else:
  print("Skip Campaign New Audience Counts FLP ")

Skip Campaign New Audience Counts FLP 


In [0]:
if pull_camp_aud == "Y": 

  spark.sql(f"""
            select audience_nm, campaign_name, exposure, lp_date, count(distinct amperity_id) as cnt_amp
            FROM AEP_camp_aud_hist_zz_v
            group by all
            order by audience_nm, campaign_name, exposure, lp_date
  """).display()

else:
  print("Skip Campaign Total Audience Counts LP Dates ")

Skip Campaign Total Audience Counts LP Dates 


In [0]:
if pull_camp_aud == "Y": 

  spark.sql(f"""
           select distinct audience_nm, campaign_name, exposure, 
          --audience_filename, 
          min(lp_date) as min_lp_date, max(lp_date) as max_lp_date
          --from AEP_camp_aud_v 
          from AEP_camp_aud_hist_zz_v 
          --where audience_filename ilike "%Guardian%" 
          group by all
          sort by campaign_name, exposure;
  """).display()

else:
  print("Skip Campaign Min/ Max LP Dates ")

audience_nm,campaign_name,exposure,min_lp_date,max_lp_date
OnStar_T2-CONCOV-63DaysofUFR,OnStar,T2-CONCOV-63DaysofUFR,2024-12-18,2025-02-11
OnStar_All_Subscribers,OnStar_All,Subscribers,2024-10-23,2025-02-11
OnStar_Buick_RemoteStart-RAPlan-LoggedInL90Days_Exposed,OnStar_Buick_RemoteStart-RAPlan-LoggedInL90Days,Exposed,2024-12-18,2025-02-11
OnStar_Buick_RemoteStart-RAPlan-LoggedInL90Days_Holdout,OnStar_Buick_RemoteStart-RAPlan-LoggedInL90Days,Holdout,2024-12-18,2025-02-11
OnStar_Buick_RemoteStart-RAPlan-NoLoggedInL90Days_Exposed,OnStar_Buick_RemoteStart-RAPlan-NoLoggedInL90Days,Exposed,2024-12-18,2025-02-11
OnStar_Buick_RemoteStart-RAPlan-NoLoggedInL90Days_Holdout,OnStar_Buick_RemoteStart-RAPlan-NoLoggedInL90Days,Holdout,2024-12-18,2025-02-11
OnStar_Buick_myBrand-NonRAPlan-or-RARenewal90Days_Exposed,OnStar_Buick_myBrand-NonRAPlan-or-RARenewal90Days,Exposed,2024-12-18,2025-02-11
OnStar_Buick_myBrand-NonRAPlan-or-RARenewal90Days_Holdout,OnStar_Buick_myBrand-NonRAPlan-or-RARenewal90Days,Holdout,2024-12-18,2025-02-11
OnStar_Buick_myBrand-RAPlan-LoggedInL90Days_Exposed,OnStar_Buick_myBrand-RAPlan-LoggedInL90Days,Exposed,2024-12-18,2025-02-11
OnStar_Buick_myBrand-RAPlan-LoggedInL90Days_Holdout,OnStar_Buick_myBrand-RAPlan-LoggedInL90Days,Holdout,2024-12-18,2025-02-11


In [0]:
# spark.sql(f"""
# -- %sql
# -- Get distinct counts for ever first LP date
# select count(distinct amperity_id) as cnt_amp, campaign_name, exposure 
# FROM AEP_camp_aud_v
# where first_lp_date >= '{FLP_date}'
# group by all order by campaign_name, exposure
# ;""").display()
# #.show(n=200, truncate=False)


In [0]:
%sql
-- --- Amperity_ID not found in any table
-- select * from AEP_camp_aud_v
-- where amperity_id = "09e6b58a-57bc-33e0-86ac-904558256eb1" 
-- order by campaign_name, lp_date asc --and row_nm = 1 limit 10;


In [0]:

if check_multi_status == "Y": 

    spark.sql(f"""
    -- %sql
    -- DROP TEMPORARY VARIABLE IF EXISTS cutoff_date;
    -- DECLARE VARIABLE cutoff_date STRING;

    -- SET VARIABLE cutoff_date = '2024-10-10';

    --------------------------------------------------------------------------------------------------------------
    ---- Check AMP_ID exposure status: Same Amp_id should only be expose or holdout, cannot be both  -------------
    --------------------------------------------------------------------------------------------------------------

    --DROP VIEW IF EXISTS AEP_amp_multi_exposure_v; create temp view AEP_amp_multi_exposure_v as ()

        select a.amperity_id, a.exposure, a.campaign_name, a.audience_nm, to_date(a.aep_export_ts) as date from AEP_camp_aud_v as a
            inner join 
                (select amperity_id, campaign_name , count(distinct exposure) as ct_status
                    FROM AEP_camp_aud_v 
                    where to_date(aep_export_ts) >= '{FLP_date}'
                    group by all having ct_status > 1
                ) as b
            on a.amperity_id = b.amperity_id and a.campaign_name = b.campaign_name
            order by a.amperity_id, a.campaign_name, a.exposure, a.aep_export_ts
    """).createOrReplaceTempView("AEP_amp_multi_exposure_v")

else: 
    print("Skip Multi Status Check ")


Skip Multi Status Check 


In [0]:

if check_multi_status == "Y": 

  spark.sql(f"""
            --------------------------------------------------------------------------------------------------------------
            ---- Export Counts for Amp_ID that have multiple Exposure Status  -------------
            --------------------------------------------------------------------------------------------------------------
            with dup_counts as
            ( 
                -- select count(distinct amperity_id) as ct_amp_multi_status, campaign_name FROM 
                --     (select amperity_id, campaign_name, count(distinct exposure) as ct_status
                --             FROM AEP_camp_aud_v
                --             where to_date(aep_export_ts) >= cutoff_date
                --             group by all having ct_status > 1)
                --     group by campaign_name

                select count(distinct amperity_id) as ct_amp_multi_status, campaign_name 
                FROM AEP_amp_multi_exposure_v
                group by campaign_name
            ),

            total_counts as 
            (
            select count(distinct amperity_id) as ct_amp_total, campaign_name from AEP_camp_aud_v group by campaign_name
            )

            -- Final query: Combine the two counts using a LEFT JOIN
            SELECT 
                a.campaign_name, 
                a.ct_amp_multi_status, 
                b.ct_amp_total,
                round(ct_amp_multi_status/ct_amp_total * 100,2) as perc_dup
            FROM 
                dup_counts as a
            LEFT JOIN 
                total_counts as b
            ON 
                a.campaign_name = b.campaign_name;
  """).display()

else: 
    print("Skip Multi Status Check ")

Skip Multi Status Check 


In [0]:
if check_multi_status == "Y": 

  spark.sql(f"""
            -------------------- Output sample Amperity_IDs for multiple status ---------------------------------------------

            select distinct amperity_id, exposure, campaign_name, audience_nm, max(date) as max_date, min(date) as min_date 
                from AEP_amp_multi_exposure_v 
            group by all 
            order by amperity_id, exposure
            limit 100;
  """).display()

else: 
    print("Skip Multi Status Check ")

Skip Multi Status Check 


# 2. Map Amperity_IDs to VIN / Account

In [0]:
%sql
------------------------------------------------------------
------------------ Not using for production ----------------
------------------------------------------------------------
-- DROP TEMPORARY VARIABLE IF EXISTS campaign_nm;
-- DROP TEMPORARY VARIABLE IF EXISTS FLP_date;
-- DECLARE VARIABLE campaign_nm STRING;
-- DECLARE VARIABLE FLP_date STRING;

-- SET VARIABLE campaign_nm = 'OnStar_Trialist';
-- SET VARIABLE FLP_date = '2024-10-10';

-- DROP VIEW IF EXISTS Campaign_v;
-- create temp view Campaign_v as 
-- (

-- with base_aud_amp as (
--   select 
--     aud.amperity_id, 
--     aud.lp_date, 
--     aud.first_lp_date, 
--     --aud.row_nm_flp,
--     aud.audience_nm, 
--     aud.campaign_name, 
--     aud.exposure, 

--     ids.account_number,
--     ids.vin, 

--     vdims.model,
--     vdims.brand,
--     vdims.model_year
--     -- vdims.google_built_in_flag,
--     -- vdims.drpo_prf_flag,
--     -- vdims.drpo_r9m_flag,
--     -- case when (vdims.drpo_prf_flag = 1 or vdims.drpo_r9m_flag = 1) then 1 else 0 end as drpo_flag

--     FROM AEP_camp_aud_v as aud

--     LEFT JOIN 
--       (
--         SELECT DISTINCT VIN, amperity_id, account_number 
--         FROM acquire.gold_connected_vehicle.member_base_history
--         --WHERE is_current_snapshot = TRUE -- It remvoe other VINs
--         ) as ids 
--       ON aud.amperity_id = ids.amperity_id

--     LEFT JOIN acquire.gold_connected_vehicle.member_base_vehicle_dim as vdims 
--       ON ids.vin = vdims.vin

--   where campaign_name in (campaign_nm) 
--   AND row_nm_flp = 1 
--   AND first_lp_date >= FLP_date
--   )

-- select a.*, cnt_account_per_amp, cnt_vin_per_amp
--   FROM base_aud_amp as a
--   LEFT JOIN (
--     select amperity_id, 
--     COUNT(DISTINCT account_number) AS cnt_account_per_amp,
--     COUNT(DISTINCT vin) AS cnt_vin_per_amp
--     FROM base_aud_amp
--     group by ALL ) as counts ON a.amperity_id = counts.amperity_id
-- );

--select * from Campaign_v order by amperity_id;

In [0]:
# %sql
# select count(distinct amperity_id) as cnt_amp, 
#   count(distinct account_number) as cnt_accnt, 
#   count(distinct vin) as cnt_vin, 
#   campaign_nm, exposure 
#   from Campaign_v 
#   group by all;

In [0]:

print(f"Campaign Selected: {campaign_nm}\n")

if campaign_nm == "'OnStar_Dormant'": 
  print("OnStar_Dormant campaign selected, filter to NonActive Accounts")

  spark.sql(f"""
  
  with base_aud_amp_dedup as (
    select distinct amperity_id, 
      --lp_date, 
      first_lp_date, 
      last_lp_date,
      row_nm_flp,
      audience_nm, 
      campaign_name, 
      exposure
    FROM AEP_camp_aud_v
    WHERE 
      last_lp_date >= '{FLP_date}'
    AND last_lp_date <= '{Max_lp_date}'
  ),

  base_aud_amp as (
    select 
      aud.amperity_id, 
      --aud.lp_date, 
      aud.first_lp_date, 
      aud.last_lp_date,
      aud.row_nm_flp,
      aud.audience_nm, 
      aud.campaign_name, 
      aud.exposure,

      acct.account_nbr as account_number,
      sub.vin,

      vin.model_nm as model,
      vin.make_nm as brand,
      vin.model_yr as model_year,

      case when (acct.account_nbr is not null and sub.vin is null) then "No VIN" -- Account number BUT no VIN
        when (acct.account_nbr is not null and sub.vin is not null) then "Both"  -- Good, both Account & VIN
        when (acct.account_nbr is null and sub.vin is null) then "Neither"       -- No Account Nor VIN matched 
        else "Unknown" end as Acct_VIN_Status                                    -- Anything else 

    FROM base_aud_amp_dedup as aud

    LEFT JOIN mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_onstar_account as acct 
      on aud.amperity_id = acct.account_owner_amperity_id
      --AND account_is_open_ind = 1
    left join mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_onstar_vehicle_subscription as sub --using(sub)
      on acct.account_nbr = sub.account_nbr
        --AND acct.account_status_nm = 'Open' 
        --AND sub.vehicle_status_cd = 'VSACTIVE'
    -- left join mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_onstar_product_subscriptions prods using(subscription_nkey)
    left join mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_vehicle as vin --using(vin)
      on vin.vin = sub.vin
    left join mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_onstar_unit as unit 
      on vin.onstar_station_id = unit.station_id
    WHERE 
      campaign_name in ({campaign_nm})
      --campaign_name in ('OnStar_Connected','OnStar_Connected_Paid','OnStar_Connected_Paid_Media')
      --AND row_nm_flp = 1 -- Removed condition since it will not align with lp_date condition
      AND first_lp_date >= '{FLP_date}' -- filter by first list pull ever date
      
      --AND account_country_cd = 'US'
      --and prods.price_plan_cd IN ('MTHLY','SUBSCBRPPD','SBSCBRPPD','COMP')
      --and prods.prod_type_cd = 'CORE'
      --and prods.prod_create_dt >= '2024-06-05'
      --and acct.account_status_nm = 'Open' 
      --and sub.vehicle_status_cd = 'VSACTIVE'
      --and unit.unit_gen_num >= 10
      --and unit.unit_state_desc IN ('Enabled','Connected')
    )

  select a.*, cnt_account_per_amp, cnt_vin_per_amp
    FROM base_aud_amp as a
    LEFT JOIN (
      select amperity_id, 
      COUNT(DISTINCT account_number) AS cnt_account_per_amp,
      COUNT(DISTINCT vin) AS cnt_vin_per_amp
      FROM base_aud_amp
      group by ALL ) as counts
        ON a.amperity_id = counts.amperity_id     
  """).createOrReplaceTempView("Campaign_dmp_v")

### Else if it's not Dormant  
else:
  spark.sql(f"""

  -- %sql
  -- DROP TEMPORARY VARIABLE IF EXISTS campaign_nm;
  -- DROP TEMPORARY VARIABLE IF EXISTS FLP_date;
  -- DECLARE VARIABLE campaign_nm STRING;
  -- DECLARE VARIABLE FLP_date STRING;

  -- SET VARIABLE campaign_nm = 'OnStar_Trialist';
  -- SET VARIABLE FLP_date = '2024-10-10';

  -- DROP VIEW IF EXISTS Campaign_dmp_v;
  -- create temp view Campaign_dmp_v as ()  

  with base_aud_amp_dedup as (
    select distinct amperity_id, 
      --lp_date, 
      first_lp_date, 
      last_lp_date,
      row_nm_flp,
      audience_nm, 
      campaign_name, 
      exposure
    FROM AEP_camp_aud_v
    WHERE 
      last_lp_date >= '{FLP_date}'
    AND last_lp_date <= '{Max_lp_date}'
  ),

  base_aud_amp as (
    select 
      aud.amperity_id, 
      --aud.lp_date, 
      aud.first_lp_date, 
      aud.row_nm_flp,
      aud.audience_nm, 
      aud.campaign_name, 
      aud.exposure,

      acct.account_nbr as account_number,
      sub.vin,

      vin.model_nm as model,
      vin.make_nm as brand,
      vin.model_yr as model_year,

      case when (acct.account_nbr is not null and sub.vin is null) then "No VIN" -- Account number BUT no VIN
        when (acct.account_nbr is not null and sub.vin is not null) then "Both"  -- Good, both Account & VIN
        when (acct.account_nbr is null and sub.vin is null) then "Neither"       -- No Account Nor VIN matched 
        else "Unknown" end as Acct_VIN_Status                                    -- Anything else 

    FROM base_aud_amp_dedup as aud

    LEFT JOIN mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_onstar_account as acct 
      on aud.amperity_id = acct.account_owner_amperity_id
      AND account_is_open_ind = 1
    left join mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_onstar_vehicle_subscription as sub --using(sub)
      on acct.account_nbr = sub.account_nbr
        AND acct.account_status_nm = 'Open' 
        AND sub.vehicle_status_cd = 'VSACTIVE'
    -- left join mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_onstar_product_subscriptions prods using(subscription_nkey)
    left join mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_vehicle as vin --using(vin)
      on vin.vin = sub.vin
    left join mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_onstar_unit as unit 
      on vin.onstar_station_id = unit.station_id
    WHERE 
      campaign_name in ({campaign_nm})
      --campaign_name in ('OnStar_Connected','OnStar_Connected_Paid','OnStar_Connected_Paid_Media')
      --AND row_nm_flp = 1 -- Removed condition since it will not align with lp_date condition 
      AND first_lp_date >= '{FLP_date}' -- filter by first list pull ever date    
      --AND account_country_cd = 'US'
      --and prods.price_plan_cd IN ('MTHLY','SUBSCBRPPD','SBSCBRPPD','COMP')
      --and prods.prod_type_cd = 'CORE'
      --and prods.prod_create_dt >= '2024-06-05'
      --and acct.account_status_nm = 'Open' 
      --and sub.vehicle_status_cd = 'VSACTIVE'
      --and unit.unit_gen_num >= 10
      --and unit.unit_state_desc IN ('Enabled','Connected')
    )

  select a.*, cnt_account_per_amp, cnt_vin_per_amp
    FROM base_aud_amp as a
    LEFT JOIN (
      select amperity_id, 
      COUNT(DISTINCT account_number) AS cnt_account_per_amp,
      COUNT(DISTINCT vin) AS cnt_vin_per_amp
      FROM base_aud_amp
      group by ALL ) as counts
        ON a.amperity_id = counts.amperity_id     
  """).createOrReplaceTempView("Campaign_dmp_v")


spark.sql(f"""
          select distinct campaign_name, exposure from Campaign_dmp_v;
""").display()
#.show(truncate=False)


print(f"Amperity_ID, Account VIN status:\n")
spark.sql(f"""
          select count(distinct amperity_id) as cnt_amp, Acct_VIN_Status from Campaign_dmp_v group by Acct_VIN_Status;
""").display()


Campaign Selected: 'OnStar_Connected', 'OnStar_Connected_Paid', 'OnStar_Connected_Paid_Media'



campaign_name,exposure
OnStar_Connected,Holdout
OnStar_Connected_Paid_Media,Exposed


Amperity_ID, Account VIN status:



cnt_amp,Acct_VIN_Status
28478,Neither
544085,Both
13570,No VIN


In [0]:
%sql
-- select a.*, b.amperity_id as Neither_amperity_id from Campaign_dmp_v as a 
--   left join (select distinct amperity_id from Campaign_dmp_v where Acct_VIN_Status = "Neither") as b
--   on a.amperity_id = b.amperity_id order by a.amperity_id, account_number, vin
--   limit 1000;


In [0]:
%sql
--select * from mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_onstar_account where account_owner_amperity_id in ("00000bf1-e50b-33df-b3fb-6d44779c9a11")

In [0]:

if fix_campaign == "Y":

  print(f"Fix campaign name for selected campaigns: {campaign_nm}\n")

  # Create df to avoid temp veiw recursive
  df = spark.sql("SELECT * FROM Campaign_dmp_v")

  from pyspark.sql import functions as F

  df_updated = df.withColumn(
    "campaign_name",
      F.when((F.col("campaign_name") == "OnStar_Connected") & (F.col("exposure") == "Holdout"), "OnStar_Connected_Paid_Media")
      .when((F.col("campaign_name") == "OnStar_Connected_Paid") & (F.col("exposure") == "Media"), "OnStar_Connected_Paid_Media")
      .otherwise(F.col("campaign_name"))
  ).withColumn(
      "exposure",
      F.when((F.col("campaign_name") == "OnStar_Connected_Paid_Media") & (F.col("exposure") == "Media"), "Exposed")
      .otherwise(F.col("exposure"))
  ).withColumn(
    "audience_nm",
      F.when((F.col("campaign_name") == "OnStar_Connected_Paid_Media") & (F.col("exposure") == "Exposed"), "OnStar_Connected_Paid_Media_Exposed")
        .when((F.col("campaign_name") == "OnStar_Connected_Paid_Media") & (F.col("exposure") == "Holdout"), "OnStar_Connected_Paid_Media_Holdout")
        .otherwise(F.col("audience_nm"))
  )

  # Recreate the Temp View
  df_updated.createOrReplaceTempView("Campaign_dmp_v")

  spark.sql("SELECT distinct audience_nm, campaign_name, exposure FROM Campaign_dmp_v").display()#.show(truncate=False)
  

else:
  print("No Need to fix for campaign name ")


Fix campaign name for selected campaigns: 'OnStar_Connected', 'OnStar_Connected_Paid', 'OnStar_Connected_Paid_Media'



audience_nm,campaign_name,exposure
OnStar_Connected_Paid_Media_Exposed,OnStar_Connected_Paid_Media,Exposed
OnStar_Connected_Paid_Media_Holdout,OnStar_Connected_Paid_Media,Holdout


In [0]:
%sql
----- Account Information Table
-- Map Amperity_ID to get Account_nbr, added filter for account_is_open_ind = 1 (active account)
--select * from mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_onstar_account where account_nbr in (136979148,154295001 )--limit 1;
--select * from mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_onstar_account where account_nbr in (146332667)-- 1 account to 10+ VINs 
--select * from mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_onstar_account where account_nbr in (162539471)-- 1 account to 10+ VINs 

-- 86 VINs
--select * from mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_onstar_account where account_owner_amperity_id in ("d82dfaad-948e-3548-9654-5e4a574a2720")

-- contains accounts with Null VINs
-- select * from mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_onstar_account where account_owner_amperity_id
-- in ("005cb95e-9e37-3951-9c04-98a0905138d1", "008b0cbb-4208-3974-8863-80d5a2e4243a") order by account_owner_amperity_id

--("001537be-3bee-31ac-ba33-fd0021e3a145")--limit 1;


-- Account, VIN, package information
-- Map account_nbr to get VIN
--select * from mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_onstar_vehicle_subscription where account_nbr in (179947349)--limit 1;
--select * from mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_onstar_vehicle_subscription where account_nbr in (146332667)-- 1 account to 10+ VINs
--select * from mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_onstar_vehicle_subscription where account_nbr in (169578732) --86 VINs
-- Amp_ID mapped to 2 Account and 1 of them has null VINs
--select * from mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_onstar_vehicle_subscription where account_nbr in (168889959,175815315)
--select * from mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_onstar_vehicle_subscription where account_nbr in (136088086,166552891)





-- Prod Package information, account_nbr, vin, subscription_nkey, subscription history
--select * from mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_onstar_product_subscriptions where account_nbr in (162539471)--limit 1; 

--select * from mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_vehicle limit 1; --VIN, vehcile infor 
--select * from mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_onstar_unit where vin in ("1GNERJKW1MJ101140", "1GNERGKS2RJ233086") --limit 1;


---------------------------------- vs. gold table status:
--select * from acquire.gold_connected_vehicle.member_base_history where account_number in (179947349)--limit 1; -- Check multi VIN status
--select * from acquire.gold_connected_vehicle.member_base_history where account_number in (162539471)--limit 1; -- Missing in Gold, but found in DMP

---------------------------------- Check Silver talbes
-- SELECT DISTINCT amperity_id, ids.account_nbr, vdims.vin, revised_model_nm, make_nm, vin_model_year_nbr, 
--   account_subscriber_status_cd,
--   vehicle_status_cd
-- FROM  acquire.silver_connected_vehicle.connected_services_customer as ids
--   LEFT join acquire.silver_connected_vehicle.account_vehicle_subscription as acct
--   on acct.account_nbr = ids.account_nbr
--   AND ids.account_subscriber_status_cd = 'CSACTIVE'
--   AND acct.vehicle_status_cd = "VSACTIVE"
--   LEFT JOIN acquire.silver_connected_vehicle.digital_vehicle as vdims
--   on vdims.vin = acct.vin
-- --where acct.account_nbr in (182351677)
-- where 
--   acct.account_nbr in (162539471)
--   --amperity_id in ("001537be-3bee-31ac-ba33-fd0021e3a145") 




# 3. Map OnStar Orders to Amp_ID

In [0]:


spark.sql(f"""

-- %sql
----- Order tables filter after FLP_Date
-- DROP VIEW IF EXISTS order_gaa_v; 
-- create temp view order_gaa_v as ()

  SELECT 
    f.VIN_ID as vin, 
    g.ACCOUNTNUMBER as account_number, 
    a.VEH_ID, 
    a.prod_id, 
    c.CREATE_MKTG_CHANNL_CD,
    date(a.create_timstm),
    date(a.START_DT), 
    date(a.VEH_PROD_END_DT), 
    a.VEH_PROD_CANCEL_DT, 
    b.PRICE_PLAN_CD, 
    e.PROD_TYPE_CD,
    a.BNDL_OFFER_ID, 
    d.BUNDLED_OFFER_CD,
    row_number() OVER (PARTITION BY ACCOUNTNUMBER, VIN_ID ORDER BY to_date(a.create_timstm)) AS row_nm_order -- get first order date
  FROM gmdataassets.dl_edge_base_gaa_14918_base_oagccxwp_gaa.ons_veh_prod a
  INNER JOIN gmdataassets.dl_edge_base_gaa_14918_base_oagccxwp_gaa.ons_order b
      ON a.order_nbr = b.order_nbr
      --AND b.PRICE_PLAN_CD in ('MTHLY','SUBSCBRPPD','COMP')   
      --AND  b.PRICE_PLAN_CD in ('MTHLY','SUBSCBRPPD','SBSCBRPPD','COMP', 'FCTRYPPD') -- Factory Prepaid
      AND  b.PRICE_PLAN_CD in ('MTHLY','SUBSCBRPPD','SBSCBRPPD','COMP') --Add SBSCBRPPD due to an update
  INNER JOIN gmdataassets.dl_edge_base_gaa_14918_base_oagccxwp_gaa.ons_order_line c
      ON a.order_nbr = c.order_nbr
      AND a.ORDER_LINE_NBR = c.ORDER_LINE_NBR
  --    AND c.CREATE_MKTG_CHANNL_CD = 'MC_ADVSR'
  INNER JOIN gmdataassets.dl_edge_base_gaa_14918_base_oagccxwp_gaa.ons_bndl_offer d
      ON a.bndl_offer_id = d.id
  INNER JOIN gmdataassets.dl_edge_base_gaa_14918_base_oagccxwp_gaa.ons_prod e
      ON a.prod_id = e.id
      AND e.PROD_TYPE_CD = 'CORE'
  INNER JOIN gmdataassets.dl_edge_base_gaa_14918_base_oagccxwp_gaa.ons_veh f
      ON a.veh_id = f.id
  INNER JOIN gmdataassets.dl_edge_base_gaa_14918_base_oagccxwp_ccsowner.r_account g
      ON a.APLCBL_ACCT_ID = g.id
      AND g.ACCOUNTTYPECODE IN ('PN','BN')
  INNER JOIN gmdataassets.dl_edge_base_gaa_14918_base_oagccxwp_ccsowner.POSTALADDRESS h
      ON f.GARAGE_ADDR_LOC_ID = h.contactinfoid
      AND h.COUNTRYISOCODE IN ('US')
  where
    date(a.create_timstm) between '{FLP_date}'  --Only looking at Orders between first_lp date to Last Conversion date
        and '{conv_end_date}'
    
""").createOrReplaceTempView("order_gaa_v")


spark.sql(f"""
select min(create_timstm) as min_date, max(create_timstm) as max_date from order_gaa_v
""").display()#.show(truncate=False)

min_date,max_date
2024-10-28,2025-02-10


In [0]:

spark.sql(f"""
-- %sql
----- Order tables filter after FLP_Date
-- DROP VIEW IF EXISTS order_gaa_hist_v;
-- create temp view order_gaa_hist_v as ()


  SELECT 
    f.VIN_ID as vin, 
    g.ACCOUNTNUMBER as account_number, 
    a.VEH_ID, 
    a.prod_id, 
    c.CREATE_MKTG_CHANNL_CD,
    date(a.create_timstm), 
    date(a.START_DT), 
    date(a.VEH_PROD_END_DT), 
    a.VEH_PROD_CANCEL_DT, 
    b.PRICE_PLAN_CD, 
    e.PROD_TYPE_CD,
    a.BNDL_OFFER_ID, 
    d.BUNDLED_OFFER_CD,
    row_number() OVER (PARTITION BY ACCOUNTNUMBER, VIN_ID ORDER BY to_date(a.create_timstm) desc ) AS row_nm_last_odr -- get first order date
  FROM gmdataassets.dl_edge_base_gaa_14918_base_oagccxwp_gaa.ons_veh_prod a
  INNER JOIN gmdataassets.dl_edge_base_gaa_14918_base_oagccxwp_gaa.ons_order b
      ON a.order_nbr = b.order_nbr
      --AND b.PRICE_PLAN_CD in ('MTHLY','SUBSCBRPPD','COMP')   
      --AND  b.PRICE_PLAN_CD in ('MTHLY','SUBSCBRPPD','SBSCBRPPD','COMP', 'FCTRYPPD') -- Factory Prepaid
      AND  b.PRICE_PLAN_CD in ('MTHLY','SUBSCBRPPD','SBSCBRPPD','COMP') --Add SBSCBRPPD due to an update
  INNER JOIN gmdataassets.dl_edge_base_gaa_14918_base_oagccxwp_gaa.ons_order_line c
      ON a.order_nbr = c.order_nbr
      AND a.ORDER_LINE_NBR = c.ORDER_LINE_NBR
  --    AND c.CREATE_MKTG_CHANNL_CD = 'MC_ADVSR'
  INNER JOIN gmdataassets.dl_edge_base_gaa_14918_base_oagccxwp_gaa.ons_bndl_offer d
      ON a.bndl_offer_id = d.id
  INNER JOIN gmdataassets.dl_edge_base_gaa_14918_base_oagccxwp_gaa.ons_prod e
      ON a.prod_id = e.id
      AND e.PROD_TYPE_CD = 'CORE'
  INNER JOIN gmdataassets.dl_edge_base_gaa_14918_base_oagccxwp_gaa.ons_veh f
      ON a.veh_id = f.id
  INNER JOIN gmdataassets.dl_edge_base_gaa_14918_base_oagccxwp_ccsowner.r_account g
      ON a.APLCBL_ACCT_ID = g.id
      AND g.ACCOUNTTYPECODE IN ('PN','BN')
  INNER JOIN gmdataassets.dl_edge_base_gaa_14918_base_oagccxwp_ccsowner.POSTALADDRESS h
      ON f.GARAGE_ADDR_LOC_ID = h.contactinfoid
      AND h.COUNTRYISOCODE IN ('US')
  where
    date(a.create_timstm) between '2022-01-01' and '{FLP_date}'  --Only looking at Orders after first_lp date
    --date(a.create_timstm) >= '2024-10-01'

""").createOrReplaceTempView("order_gaa_hist_v")

spark.sql(f"""
select min(create_timstm) as min_date, max(create_timstm) as max_date from order_gaa_hist_v
""").display()#.show(truncate=False)

min_date,max_date
2022-01-01,2024-10-28


In [0]:
spark.sql(f"""
-- %sql

-------------------- Create Temp table for history Orders --------------------------------

-- DROP VIEW IF EXISTS amp_order_hist_v;
-- CREATE temp view amp_order_hist_v as ( )

  SELECT aud.*,
    ord.row_nm_last_odr,
  
    datediff(month,create_timstm, '{FLP_date}') as latest_purchase_months,
    CASE WHEN datediff(month,create_timstm, '{FLP_date}') is null then 999
      WHEN datediff(month,create_timstm,'{FLP_date}') >= 0 and datediff(month,create_timstm, '{FLP_date}') <=3 then '<3 Month' 
      WHEN datediff(month,create_timstm, '{FLP_date}') >=4 and datediff(month,create_timstm, '{FLP_date}') <=6  then '4 - 6 Month'
      WHEN datediff(month,create_timstm, '{FLP_date}') >=7 and datediff(month,create_timstm, '{FLP_date}') <=12  then '7 - 12 Month'
      WHEN datediff(month,create_timstm, '{FLP_date}') >=13 and datediff(month,create_timstm, '{FLP_date}') <=18  then '1YR - 1YR6M'
      WHEN datediff(month,create_timstm, '{FLP_date}') >=19 and datediff(month,create_timstm, '{FLP_date}') <=24  then '1YR7M - 2YR'
      WHEN datediff(month,create_timstm,'{FLP_date}') >=25 then '>2YR'
      ELSE 'NA' 
      END AS latest_purchase_group

  FROM Campaign_dmp_v as aud 
  LEFT JOIN order_gaa_hist_v as ord 
    ON aud.vin = ord.vin 
    AND aud.account_number = ord.account_number
    AND row_nm_last_odr = 1
    WHERE Acct_VIN_Status not in ("No VIN") -- Remove conditions: Missing VIN only (with accoutn number)
  --WHERE Acct_VIN_Status not in ("No VIN", "Neither") -- Remove conditions: Missing VIN, Missing neither VIN & Account_Number
""").createOrReplaceTempView("amp_order_hist_v")


spark.sql(f"""
select * from amp_order_hist_v limit 10
""").display()#.show(truncate=False)

amperity_id,first_lp_date,row_nm_flp,audience_nm,campaign_name,exposure,account_number,vin,model,brand,model_year,Acct_VIN_Status,cnt_account_per_amp,cnt_vin_per_amp,row_nm_last_odr,latest_purchase_months,latest_purchase_group
00564efe-28e2-335b-9e22-5fd2c2508d69,2025-01-17,1,OnStar_Connected_Paid_Media_Exposed,OnStar_Connected_Paid_Media,Exposed,161258466,1G1BC5SMXK7129878,Cruze,Chevrolet,2019,Both,1,1,null,null,999
000f6354-4cf4-3b54-b4e5-5579022c5f86,2025-01-17,1,OnStar_Connected_Paid_Media_Exposed,OnStar_Connected_Paid_Media,Exposed,173392541,2GNALAEK2F1174796,Equinox,Chevrolet,2015,Both,1,1,1,31,>2YR
000af35c-28b9-3c19-b7c8-df4bec97dc56,2024-11-11,1,OnStar_Connected_Paid_Media_Exposed,OnStar_Connected_Paid_Media,Exposed,170493822,3GKALVEG3PL138955,Terrain,GMC,2023,Both,1,1,1,9,7 - 12 Month
000af35c-28b9-3c19-b7c8-df4bec97dc56,2024-11-11,2,OnStar_Connected_Paid_Media_Exposed,OnStar_Connected_Paid_Media,Exposed,170493822,3GKALVEG3PL138955,Terrain,GMC,2023,Both,1,1,1,9,7 - 12 Month
000af35c-28b9-3c19-b7c8-df4bec97dc56,2024-11-11,3,OnStar_Connected_Paid_Media_Exposed,OnStar_Connected_Paid_Media,Exposed,170493822,3GKALVEG3PL138955,Terrain,GMC,2023,Both,1,1,1,9,7 - 12 Month
00216efb-bedc-3b2e-a14d-fe7f4105a146,2025-01-17,1,OnStar_Connected_Paid_Media_Holdout,OnStar_Connected_Paid_Media,Holdout,170246796,1G1BC5SM2J7190348,Cruze,Chevrolet,2018,Both,1,2,1,4,4 - 6 Month
0027783f-afee-3c86-8069-39c04879b8a7,2025-01-17,1,OnStar_Connected_Paid_Media_Exposed,OnStar_Connected_Paid_Media,Exposed,4187878,1GCRCSEC0HZ298764,Silverado,Chevrolet,2017,Both,1,1,null,null,999
000d7b51-e4a5-3aa1-af1d-1338796f3658,2025-01-17,1,OnStar_Connected_Paid_Media_Holdout,OnStar_Connected_Paid_Media,Holdout,172441569,1GNSCJKC0FR741550,Suburban,Chevrolet,2015,Both,1,1,null,null,999
00114032-d787-3799-a55a-91d58283b4a4,2025-01-17,1,OnStar_Connected_Paid_Media_Holdout,OnStar_Connected_Paid_Media,Holdout,167771274,3GCUKREC9HG242148,Silverado,Chevrolet,2017,Both,1,1,null,null,999
001fd9a0-a55b-39c2-88a9-4d5f8d11f4ae,2025-01-17,1,OnStar_Connected_Paid_Media_Exposed,OnStar_Connected_Paid_Media,Exposed,147856406,3GCUYBEF7KG700079,Silverado,Chevrolet,2019,Both,1,2,1,4,4 - 6 Month


In [0]:
%sql
-- select * from mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_onstar_product_subscriptions 
--   where prod_create_dt > "2024-12-31"
--   limit 100;

--select distinct prod_sub_status_desc from mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_onstar_product_subscriptions ;


-- prod_sub_status_desc
-- Future
-- Cancelled
-- Active
-- Void
-- Expired
-- VOID



-- select *, 
--       row_number() OVER (PARTITION BY vin, account_nbr ORDER BY prod_create_dt) AS row_nm_order
--       from mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_onstar_product_subscriptions
--       where to_date(prod_create_dt) >= FLP_date
--       AND PRICE_PLAN_CD in ('MTHLY','SUBSCBRPPD','SBSCBRPPD','COMP')
--       AND PROD_TYPE_CD = 'CORE'
--       order by account_nbr, vin, prod_create_dt
--       limit 1000;

In [0]:
%sql
-- select * from order_gaa_v 
-- --where ACCOUNTNUMBER in (162539471)
-- --where vin in ("1GTP6BEK4R1113587")
-- order by account_number, vin, create_timstm
-- limit 100;

# 4. Check Distribution

In [0]:
%sql
select count(distinct amperity_id) as cnt_amp, 
  count(distinct account_number) as cnt_accnt, 
  count(distinct vin) as cnt_vin, 
  round(count(distinct account_number) / count(distinct amperity_id),3) as avg_accnt_per_amp,
  round(count(distinct vin) / count(distinct amperity_id),3) as avg_vin_per_amp,
  campaign_name, exposure 
  from Campaign_dmp_v 
  group by all

union all 

select count(distinct amperity_id) as cnt_amp, 
  count(distinct account_number) as cnt_accnt, 
  count(distinct vin) as cnt_vin, 
  round(count(distinct account_number) / count(distinct amperity_id),3) as avg_accnt_per_amp,
  round(count(distinct vin) / count(distinct amperity_id),3) as avg_vin_per_amp,
  campaign_name, "All" as exposure 
  from Campaign_dmp_v 
  group by all

cnt_amp,cnt_accnt,cnt_vin,avg_accnt_per_amp,avg_vin_per_amp,campaign_name,exposure
146168,152124,175795,1.041,1.203,OnStar_Connected_Paid_Media,Holdout
430383,439761,500752,1.022,1.164,OnStar_Connected_Paid_Media,Exposed
573424,587477,665270,1.025,1.16,OnStar_Connected_Paid_Media,All


In [0]:
%sql

-- select * from order_gaa_hist_v order by account_number, vin, create_timstm, row_nm_last_odr
-- limit 100;


-------------------- Output for Varias Check --------------------------------

select count(distinct amperity_id) as cnt_amp, 
  count(distinct account_number) as cnt_accnt, 
  count(distinct vin) as cnt_vin, 
  campaign_name, exposure, cnt_account_per_amp as Metric, "cnt_account_per_amp"  as Metric_Type
  from Campaign_dmp_v 
  WHERE Acct_VIN_Status not in ("No VIN") -- Remove conditions: Missing VIN
  --where Acct_VIN_Status not in ("No VIN", "Neither")
  group by all

UNION ALL

select count(distinct amperity_id) as cnt_amp, 
  count(distinct account_number) as cnt_accnt, 
  count(distinct vin) as cnt_vin, 
  campaign_name, exposure, cnt_vin_per_amp as Metric, "cnt_vin_per_amp"  as Metric_Type
  from Campaign_dmp_v 
  WHERE Acct_VIN_Status not in ("No VIN") -- Remove conditions: Missing VIN
  --where Acct_VIN_Status not in ("No VIN", "Neither")
  group by all

UNION ALL

select count(distinct amperity_id) as cnt_amp, 
  count(distinct account_number) as cnt_accnt, 
  count(distinct vin) as cnt_vin, 
  campaign_name, exposure, brand as Metric, "Brand"  as Metric_Type
  from Campaign_dmp_v
  WHERE Acct_VIN_Status not in ("No VIN") -- Remove conditions: Missing VIN 
  --where Acct_VIN_Status not in ("No VIN", "Neither")
  group by all

UNION ALL

select count(distinct amperity_id) as cnt_amp, 
  count(distinct account_number) as cnt_accnt, 
  count(distinct vin) as cnt_vin, 
  campaign_name, exposure, model_year as Metric, "Model Year"  as Metric_Type
  from Campaign_dmp_v 
  WHERE Acct_VIN_Status not in ("No VIN") -- Remove conditions: Missing VIN
  --where Acct_VIN_Status not in ("No VIN", "Neither")
  group by all


UNION ALL

select count(distinct amperity_id) as cnt_amp, 
  count(distinct account_number) as cnt_accnt, 
  count(distinct vin) as cnt_vin, 
  campaign_name, exposure, model as Metric, "Model"  as Metric_Type
  from Campaign_dmp_v 
  WHERE Acct_VIN_Status not in ("No VIN") -- Remove conditions: Missing VIN
  --where Acct_VIN_Status not in ("No VIN", "Neither")
  group by all

UNION ALL

select count(distinct amperity_id) as cnt_amp, 
  count(distinct account_number) as cnt_accnt, 
  count(distinct vin) as cnt_vin, 
  campaign_name, exposure, latest_purchase_months as Metric, "Latest Purchase Months"  as Metric_Type
  from amp_order_hist_v 
  WHERE Acct_VIN_Status not in ("No VIN") -- Remove conditions: Missing VIN
  --where Acct_VIN_Status not in ("No VIN", "Neither")
  group by all

UNION ALL

select count(distinct amperity_id) as cnt_amp, 
  count(distinct account_number) as cnt_accnt, 
  count(distinct vin) as cnt_vin, 
  campaign_name, exposure, latest_purchase_group as Metric, "Latest Purchase Groups"  as Metric_Type
  from amp_order_hist_v
  WHERE Acct_VIN_Status not in ("No VIN") -- Remove conditions: Missing VIN 
  --where Acct_VIN_Status not in ("No VIN", "Neither")
  group by all


  sort by exposure, Metric_Type, Metric


cnt_amp,cnt_accnt,cnt_vin,campaign_name,exposure,Metric,Metric_Type
21574,0,0,OnStar_Connected_Paid_Media,Exposed,0,cnt_account_per_amp
382025,382025,431469,OnStar_Connected_Paid_Media,Exposed,1,cnt_account_per_amp
4,29,45,OnStar_Connected_Paid_Media,Exposed,10,cnt_account_per_amp
4,40,84,OnStar_Connected_Paid_Media,Exposed,11,cnt_account_per_amp
1,9,12,OnStar_Connected_Paid_Media,Exposed,12,cnt_account_per_amp
2,15,32,OnStar_Connected_Paid_Media,Exposed,13,cnt_account_per_amp
1,10,25,OnStar_Connected_Paid_Media,Exposed,14,cnt_account_per_amp
1,8,40,OnStar_Connected_Paid_Media,Exposed,142,cnt_account_per_amp
2,18,21,OnStar_Connected_Paid_Media,Exposed,15,cnt_account_per_amp
2,30,37,OnStar_Connected_Paid_Media,Exposed,16,cnt_account_per_amp


# 5. Output - Amp - GAA Orders

In [0]:
%sql

DROP VIEW IF EXISTS amp_order_v;
CREATE temp view amp_order_v as (

with temp_base as (  
  SELECT aud.*,
    row_nm_order,
    CREATE_MKTG_CHANNL_CD,
    prod_id,
    PRICE_PLAN_CD, 
    PROD_TYPE_CD,
    to_date(create_timstm) as purchase_date,
    datediff(purchase_date, first_lp_date) as purchase_flp,
    
    -- Define prior_purchase if purchased happend prior FLP for each
    CASE WHEN purchase_date < first_lp_date and PRICE_PLAN_CD in ('COMP') then -1 
      WHEN purchase_date < first_lp_date and PRICE_PLAN_CD <> ('COMP') then 1 
      WHEN  purchase_date >= first_lp_date then 0 
      WHEN purchase_date is null then NULL
      else 999 END as prior_purchase

  FROM Campaign_dmp_v as aud 
    LEFT JOIN order_gaa_v as ord 
    ON aud.vin = ord.vin 
    AND aud.account_number = ord.account_number
  WHERE Acct_VIN_Status not in ("No VIN") -- Remove conditions: Missing VIN  
  --WHERE Acct_VIN_Status not in ("No VIN", "Neither")
  )

select a.*, 
  max_prior_purchase,
  min_prior_purchase,
  
  CASE WHEN prior_purchase = -1 and max_prior_purchase = -1 THEN 0 -- Prior purchased COMP and no future Pruchased
    WHEN min_prior_purchase = -1 and max_prior_purchase = 1 THEN 0 -- Prior purchased COMP and Other Packages | USE MIN & MAX
    ELSE max_prior_purchase
    END as prior_purchase_status,

  --Row Number subscriptions based on prior_purchase status, will count first purchase for prior and post status
  row_number() OVER (PARTITION BY a.amperity_id, a.account_number, a.vin, a.prior_purchase order by purchase_date) as row_nm_order_prior,
  --Row Number subscriptions based on purchase_date, count 1 purchase per day, all purchases will be counted
  row_number() OVER (PARTITION BY a.amperity_id, a.account_number, a.vin, purchase_date order by prod_id) as row_nm_order_per_day

  FROM temp_base as a
  LEFT JOIN 
    (select amperity_id, account_number, vin, audience_nm, 
      max(prior_purchase) as max_prior_purchase,
      min(prior_purchase) as min_prior_purchase
    FROM temp_base group by ALL
    ) as b 
  ON a.amperity_id = b.amperity_id 
  and a.account_number = b.account_number 
  and a.vin = b.vin 
  and a.audience_nm = b.audience_nm 
  
  WHERE Acct_VIN_Status not in ("No VIN") -- Remove conditions: Missing VIN
  --WHERE Acct_VIN_Status not in ("No VIN", "Neither")
);




--- Filter to ever first order for prior and post purchase status
DROP VIEW IF EXISTS amp_order_first_v;
create temp view amp_order_first_v as (
  SELECT * FROM amp_order_v 
    where row_nm_order_prior = 1
);


--- Filter to all order, only 1st order per day
DROP VIEW IF EXISTS amp_order_all_v;
create temp view amp_order_all_v as (
  SELECT * FROM amp_order_v 
    where row_nm_order_per_day = 1
);





In [0]:
%sql
--------------- Check table output ---------------------------
-- select * from amp_order_first_v 
--   WHERE purchase_date is not null 
--   ORDER BY amperity_id, account_number, VIN, purchase_date, row_nm_order limit 1000;


-- select * from amp_order_all_v 
--   WHERE purchase_date is not null 
--   ORDER BY amperity_id, account_number, VIN, purchase_date, row_nm_order limit 10;

--select * from order_v where account_number in (182817319)

In [0]:
%sql

---------------------------------------- Prior Purchase Status -----------------------

-- select * from amp_order_v 
--   where amperity_id in (
--     select distinct amperity_id from amp_order_v 
--     where prior_purchase in (-1) 
--       and max_prior_purchase = 1
--   )
-- order by amperity_id, account_number, vin, prior_purchase, purchase_date limit 1000;

-- Notes
-- Prior_purchase = -1, purchased COMP before
-- max(status) = -1: Prior purchased COMP only, no other purchases or future purchases
-- max(status) = 0: Prior purchased COMP and Purchased after
-- max(status) = 1: Prior purchased COMP and OTHER Purchase Prior

------------------------------------------------------------------------------------------

In [0]:

################################## Prior Purchase Detail Distribution  ####################################

df_prior_purchase_detail = spark.sql(f"""

--DROP VIEW IF EXISTS prior_purchase_detail_v;
--create temp view prior_purchase_detail_v as ()
  select campaign_name, exposure, prior_purchase, max_prior_purchase, min_prior_purchase, prior_purchase_status, count(distinct amperity_id) as cnt_amp 
    from amp_order_v
    group by all
    order by campaign_name, exposure, prior_purchase, prior_purchase_status
""").toPandas()

################################## Prior Purchase Distribution - Used for Conversion Rate ####################################


df_prior_purchase = spark.sql(f"""
--DROP VIEW IF EXISTS prior_purchase_v;
--create temp view prior_purchase_v as ()

  select campaign_name, exposure, prior_purchase_status, count(distinct amperity_id) as cnt_amp 
    from amp_order_v
    group by all
    order by campaign_name, exposure, prior_purchase_status

""").toPandas()


print(f"Prior Purchase: \n")
df_prior_purchase.display()

print(f"Prior Purchase Detail: \n")
df_prior_purchase_detail.display()

#df_prior_purchase_detail.display()
#df_prior_purchase.display()


Prior Purchase: 



campaign_name,exposure,prior_purchase_status,cnt_amp
OnStar_Connected_Paid_Media,Exposed,null,417469
OnStar_Connected_Paid_Media,Exposed,0.0,13023
OnStar_Connected_Paid_Media,Exposed,1.0,6051
OnStar_Connected_Paid_Media,Holdout,null,141796
OnStar_Connected_Paid_Media,Holdout,0.0,4531
OnStar_Connected_Paid_Media,Holdout,1.0,2113


Prior Purchase Detail: 



campaign_name,exposure,prior_purchase,max_prior_purchase,min_prior_purchase,prior_purchase_status,cnt_amp
OnStar_Connected_Paid_Media,Exposed,null,null,null,null,417469
OnStar_Connected_Paid_Media,Exposed,-1.0,1.0,-1.0,0.0,1031
OnStar_Connected_Paid_Media,Exposed,-1.0,0.0,-1.0,0.0,65
OnStar_Connected_Paid_Media,Exposed,-1.0,-1.0,-1.0,0.0,1050
OnStar_Connected_Paid_Media,Exposed,0.0,0.0,-1.0,0.0,65
OnStar_Connected_Paid_Media,Exposed,0.0,1.0,-1.0,0.0,64
OnStar_Connected_Paid_Media,Exposed,0.0,0.0,0.0,0.0,10939
OnStar_Connected_Paid_Media,Exposed,0.0,1.0,0.0,1.0,368
OnStar_Connected_Paid_Media,Exposed,1.0,1.0,-1.0,0.0,1031
OnStar_Connected_Paid_Media,Exposed,1.0,1.0,0.0,1.0,368


In [0]:


df_order_first = spark.sql(f"""

--%sql
----- Output summary table for Lift calculation
select purchase_date, first_lp_date, purchase_flp, 
  campaign_name, 
  exposure, 
  prior_purchase_status,
  case when purchase_date is not null then 1 else 0 END as purchase,
  count(distinct amperity_id) as cnt_amp
from amp_order_first_v
group by all
order by purchase_date
""").toPandas()

print(f"Output First Order Only: \n")
df_order_first.display()


Output First Order Only: 



purchase_date,first_lp_date,purchase_flp,campaign_name,exposure,prior_purchase_status,purchase,cnt_amp
null,2025-01-17,null,OnStar_Connected_Paid_Media,Holdout,null,0,118572
null,2024-11-04,null,OnStar_Connected_Paid_Media,Exposed,null,0,17957
null,2024-10-28,null,OnStar_Connected_Paid_Media,Exposed,null,0,16340
null,2024-11-18,null,OnStar_Connected_Paid_Media,Exposed,null,0,14146
null,2024-10-28,null,OnStar_Connected_Paid_Media,Holdout,null,0,5572
null,2024-11-11,null,OnStar_Connected_Paid_Media,Exposed,null,0,17825
null,2025-01-17,null,OnStar_Connected_Paid_Media,Exposed,null,0,350590
null,2024-11-04,null,OnStar_Connected_Paid_Media,Holdout,null,0,6178
null,2024-11-11,null,OnStar_Connected_Paid_Media,Holdout,null,0,6027
null,2024-11-18,null,OnStar_Connected_Paid_Media,Holdout,null,0,4800


In [0]:

df_order_multi = spark.sql(f"""
--%sql

----- Output summary table for Lift calculation
select purchase_date, first_lp_date, purchase_flp, 
  campaign_name, 
  exposure, 
  prior_purchase_status,
  case when purchase_date is not null then 1 else 0 END as purchase,
  count(distinct amperity_id) as cnt_amp
from amp_order_all_v
group by all
order by purchase_date
""").toPandas()

print(f"Output Multiple Order: \n")
df_order_multi.display()

Output Multiple Order: 



purchase_date,first_lp_date,purchase_flp,campaign_name,exposure,prior_purchase_status,purchase,cnt_amp
null,2025-01-17,null,OnStar_Connected_Paid_Media,Holdout,null,0,118576
null,2024-11-04,null,OnStar_Connected_Paid_Media,Exposed,null,0,17960
null,2024-10-28,null,OnStar_Connected_Paid_Media,Exposed,null,0,16341
null,2024-11-18,null,OnStar_Connected_Paid_Media,Exposed,null,0,14145
null,2024-10-28,null,OnStar_Connected_Paid_Media,Holdout,null,0,5567
null,2024-11-11,null,OnStar_Connected_Paid_Media,Exposed,null,0,17816
null,2025-01-17,null,OnStar_Connected_Paid_Media,Exposed,null,0,350602
null,2024-11-04,null,OnStar_Connected_Paid_Media,Holdout,null,0,6178
null,2024-11-11,null,OnStar_Connected_Paid_Media,Holdout,null,0,6016
null,2024-11-18,null,OnStar_Connected_Paid_Media,Holdout,null,0,4799


# DO NOT USE - DMP Prod Tables

In [0]:
%sql

---- Not used


-- DROP VIEW IF EXISTS amp_order_v;
-- create temp view amp_order_v as (
--   SELECT aud.*,
--     to_date(ord.prod_create_dt) as purchase_date,
--     datediff(purchase_date, first_lp_date) as date_diff
--     FROM Campaign_dmp_v as aud 
--     LEFT JOIN 
--     (select *, 
--       row_number() OVER (PARTITION BY vin, account_nbr ORDER BY prod_create_dt) AS row_nm_order
--       from mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_onstar_product_subscriptions
--       where to_date(prod_create_dt) >= FLP_date
--       AND PRICE_PLAN_CD in ('MTHLY','SUBSCBRPPD','SBSCBRPPD','COMP')
--       AND PROD_TYPE_CD = 'CORE'
--       --AND prod_sub_status_desc NOT IN ('Future','Void','VOID')
--     ) as ord 
--     ON aud.vin = ord.vin 
--     AND aud.account_number = ord.account_nbr
--     AND row_nm_order = 1
-- );

-- select * from amp_order_v where purchase_date >= "2025-01-01"  limit 1000;

-- select purchase_date, first_lp_date, date_diff, campaign_name, exposure,
--   case when purchase_date is not null then 1 else 0 END as purchase,
--   count(distinct amperity_id) as cnt_amp
-- from amp_order_v
-- group by all
-- order by purchase_date;


In [0]:
%sql
--select * from amp_order_v where purchase_date > "2024-12-31" limit 100;

--select * from mktg_dmp_silver_prod.a186354_mw_vehicle.mktg_base_onstar_product_subscriptions limit 10


# 6. Measurment

In [0]:
df_prior_purchase.head()
#df_order_first.head()
#df_order_multi.head()


,campaign_name,exposure,prior_purchase_status,cnt_amp
0,OnStar_Connected_Paid_Media,Exposed,NaN,417469
1,OnStar_Connected_Paid_Media,Exposed,0.0000,13023
2,OnStar_Connected_Paid_Media,Exposed,1.0000,6051
3,OnStar_Connected_Paid_Media,Holdout,NaN,141796
4,OnStar_Connected_Paid_Media,Holdout,0.0000,4531


In [0]:

################################### Prepare Base Total ##############################################

# Condition 1: Sum all Exposed and Holdout
df_base = df_prior_purchase
base_total = df_base.groupby('exposure')['cnt_amp'].sum().copy().reset_index()

# Condition 2: Exclude prior_purchase_status = 1.0
no_piror_df = df_base[df_base['prior_purchase_status'] != 1].copy().reset_index()
base_total_no_prior = no_piror_df.groupby('exposure')['cnt_amp'].sum().copy().reset_index()

# Display results
print("Base Total:")
print(base_total.head())

print("\nBase Total: (remove prior_purchase_status = 1):")
print(base_total_no_prior.head())



Base Total:
  exposure  cnt_amp
0  Exposed  436543 
1  Holdout  148440 

Base Total: (remove prior_purchase_status = 1):
  exposure  cnt_amp
0  Exposed  430492 
1  Holdout  146327 


In [0]:

# Move to the Widget Part

#Assume df_order_first and df_order_multi are already defined DataFrames#


if df_conv_input == "First Order Only":
    df_conv_base = df_order_first
    print("First Order Selected: df_order_first")
elif df_conv_input == "Multiple Orders":
    df_conv_base = df_order_multi
    print("Multipe Order Selected: df_order_multi")
else:
    raise ValueError("Invalid lift_conv_type selected in the widget")


Multipe Order Selected: df_order_multi


In [0]:

################################### Prepare Conversion for each condition ##############################################

print(f"Selected Order Type: {df_conv_input}")


if X_days != 0: 
  #filter_date = '2024-10-25'
  filter_date = conv_start_date
  df_conversion = df_conv_base.copy().reset_index()
  df_conversion['purchase_date'] = pd.to_datetime(df_conversion['purchase_date'])
  df_conversion = df_conversion[df_conversion['purchase_date'] >= filter_date]
  # Create addition dataset for condition without priors
  df_conversion_no_prior = df_conversion[df_conversion['prior_purchase_status'] != 1 ]
  print(f"\nFilter Conversions, start from {filter_date} to {conv_end_date}")
  print(f"\nConversions Range, start from {conv_start_range_date} to {conv_end_date}")

else: 
  print(f"\nNo Filter on Conversions, start from {FLP_date} to {conv_end_date}")
  print(f"\nConversions Range, start from {conv_start_range_date} to {conv_end_date}")
  filter_date = conv_start_date
  df_conversion = df_conv_base.copy().reset_index()
  df_conversion['purchase_date'] = pd.to_datetime(df_conversion['purchase_date'])
  # Create addition dataset for condition without priors
  df_conversion_no_prior = df_conversion[df_conversion['prior_purchase_status'] != 1 ]


# Define cumulative conditions as a dictionary with an additional condition `>= 0`
conditions = {
    "<= 7 Days": df_conversion[(df_conversion['purchase_flp'] >= 0) & (df_conversion['purchase_flp'] <= 7)],
    "<= 14 Days": df_conversion[(df_conversion['purchase_flp'] >= 0) & (df_conversion['purchase_flp'] <= 14)],
    "<= 21 Days": df_conversion[(df_conversion['purchase_flp'] >= 0) & (df_conversion['purchase_flp'] <= 21)],
    "<= 30 Days": df_conversion[(df_conversion['purchase_flp'] >= 0) & (df_conversion['purchase_flp'] <= 30)],
    "Conversion Range": df_conversion[(df_conversion['purchase_date'] >= conv_start_range_date) 
                                      & (df_conversion['purchase_date'] <= conv_end_date)],
}

conditions2 = {
    "<= 7 Days": df_conversion_no_prior[(df_conversion_no_prior['purchase_flp'] >= 0) & (df_conversion_no_prior['purchase_flp'] <= 7)],
    "<= 14 Days": df_conversion_no_prior[(df_conversion_no_prior['purchase_flp'] >= 0) & (df_conversion_no_prior['purchase_flp'] <= 14)],
    "<= 21 Days": df_conversion_no_prior[(df_conversion_no_prior['purchase_flp'] >= 0) & (df_conversion_no_prior['purchase_flp'] <= 21)],
    "<= 30 Days": df_conversion_no_prior[(df_conversion_no_prior['purchase_flp'] >= 0) & (df_conversion_no_prior['purchase_flp'] <= 30)],
    "Conversion Range": df_conversion[(df_conversion['purchase_date'] >= conv_start_range_date) 
                                      & (df_conversion['purchase_date'] <= conv_end_date)],
}


# Initialize an empty list to store results
results = []
results2 = []

# Loop through conditions and calculate sums
for label, condition_df in conditions.items():
    grouped = condition_df.groupby('exposure', observed=False)['cnt_amp'].sum().reset_index()
    grouped['purchase_flp_bin'] = label  # Add the bin label
    results.append(grouped)

# Combine all results into a single DataFrame
conversion_condition = pd.concat(results, ignore_index=True)

# Loop through conditions and calculate sums
for label, condition_df in conditions2.items():
    grouped = condition_df.groupby('exposure', observed=False)['cnt_amp'].sum().reset_index()
    grouped['purchase_flp_bin'] = label  # Add the bin label
    results2.append(grouped)

# Combine all results into a single DataFrame
conversion_condition2 = pd.concat(results2, ignore_index=True)


print(f"\nInput Table: \n{conversion_condition}")
print(f"\nInput Table No Prior Purchase: \n{conversion_condition2}")

Selected Order Type: Multiple Orders

No Filter on Conversions, start from 2024-10-28 to 2025-02-10

Conversions Range, start from 2025-01-01 00:00:00 to 2025-02-10

Input Table: 
  exposure  cnt_amp  purchase_flp_bin
0  Exposed   2875           <= 7 Days
1  Holdout    986           <= 7 Days
2  Exposed   4972          <= 14 Days
3  Holdout   1690          <= 14 Days
4  Exposed   6759          <= 21 Days
5  Holdout   2356          <= 21 Days
6  Exposed   8354          <= 30 Days
7  Holdout   2910          <= 30 Days
8  Exposed   7262    Conversion Range
9  Holdout   2520    Conversion Range

Input Table No Prior Purchase: 
  exposure  cnt_amp  purchase_flp_bin
0  Exposed   2746           <= 7 Days
1  Holdout    935           <= 7 Days
2  Exposed   4770          <= 14 Days
3  Holdout   1602          <= 14 Days
4  Exposed   6492          <= 21 Days
5  Holdout   2245          <= 21 Days
6  Exposed   8036          <= 30 Days
7  Holdout   2779          <= 30 Days
8  Exposed   7262    Conver

In [0]:
# Define Base for lift analysis
df_lift_base = base_total

# Measurement conditions
conditions = ["<= 7 Days", "<= 14 Days", "<= 21 Days", "<= 30 Days","Conversion Range"]

# Initialize results list
results_Z = []


for condition in conditions:
    # Filter data for the current condition
    
    exposed_conversion = conversion_condition[(conversion_condition['purchase_flp_bin'] == condition) & (conversion_condition['exposure'] != 'Holdout')]['cnt_amp'].values[0]
    holdout_conversion = conversion_condition[(conversion_condition['purchase_flp_bin'] == condition) & (conversion_condition['exposure'] == 'Holdout')]['cnt_amp'].values[0]
    
    # Get base totals
    exposed_total = df_lift_base[df_lift_base['exposure'] != 'Holdout']['cnt_amp'].values[0]
    holdout_total = df_lift_base[df_lift_base['exposure'] == 'Holdout']['cnt_amp'].values[0]
    
    # Calculate conversion rates
    conv_exposed = exposed_conversion / exposed_total * 100
    conv_holdout = holdout_conversion / holdout_total * 100
    
    # Calculate Lift
    lift = ((conv_exposed - conv_holdout) / conv_holdout) * 100 if conv_holdout > 0 else None
    
    # Perform z-test for statistical significance
    z_stat, p_value =  sm.stats.proportions_ztest([exposed_conversion, holdout_conversion], [exposed_total, holdout_total])

    # Append results
    results_Z.append({
        'Campaign Name': df_order_first['campaign_name'].iloc[0],
        'Measurement Method': 'Z-Test',
        'Meansurement Run Date': datetime.today().strftime('%Y-%m-%d'),
        'Campaign Start Date': FLP_date,
        'List Pull End Date': last_listpull_date,  #Max_lp_date,
        #'Conversion Start Date': filter_date.strftime('%Y-%m-%d'),
        'Conversion Start Date': (conv_start_range_date if condition == "Conversion Range" else filter_date).strftime('%Y-%m-%d'),
        'Conversion End Date': conv_end_date_dt.strftime('%Y-%m-%d'),
        'Condition': condition,
        'Conversion Rate - Exposed': f'{conv_exposed:.4f}%',
        'Conversion Rate - Holdout': f'{conv_holdout:.4f}%',
        'Lift (%)': f'{lift:.4f}%',
        'P-Value': f'{p_value:.4f}',
        'Exposed - Conversion': f'{exposed_conversion:,}',
        'Exposed Total': f'{exposed_total:,}',
        'Holdout - Conversion': f'{holdout_conversion:,}',
        'Holdout Total': f'{holdout_total:,}',
    })

# Convert results to DataFrame
results_z_test_df = pd.DataFrame(results_Z)

# Display results
print("Results: Z-Test")
results_z_test_df.display()


Results: Z-Test


Campaign Name,Measurement Method,Meansurement Run Date,Campaign Start Date,List Pull End Date,Conversion Start Date,Conversion End Date,Condition,Conversion Rate - Exposed,Conversion Rate - Holdout,Lift (%),P-Value,Exposed - Conversion,Exposed Total,Holdout - Conversion,Holdout Total
OnStar_Connected_Paid_Media,Z-Test,2025-02-12,2024-10-28,2025-01-17,2024-10-28,2025-02-10,<= 7 Days,0.6586%,0.6642%,-0.8518%,0.8161,"2,875","436,543",986,"148,440"
OnStar_Connected_Paid_Media,Z-Test,2025-02-12,2024-10-28,2025-01-17,2024-10-28,2025-02-10,<= 14 Days,1.1389%,1.1385%,0.0388%,0.9890,"4,972","436,543","1,690","148,440"
OnStar_Connected_Paid_Media,Z-Test,2025-02-12,2024-10-28,2025-01-17,2024-10-28,2025-02-10,<= 21 Days,1.5483%,1.5872%,-2.4491%,0.2962,"6,759","436,543","2,356","148,440"
OnStar_Connected_Paid_Media,Z-Test,2025-02-12,2024-10-28,2025-01-17,2024-10-28,2025-02-10,<= 30 Days,1.9137%,1.9604%,-2.3830%,0.2579,"8,354","436,543","2,910","148,440"
OnStar_Connected_Paid_Media,Z-Test,2025-02-12,2024-10-28,2025-01-17,2025-01-01,2025-02-10,Conversion Range,1.6635%,1.6977%,-2.0105%,0.3757,"7,262","436,543","2,520","148,440"


In [0]:
# Define Base for lift analysis
df_lift_base = base_total_no_prior

# Measurement conditions
conditions = ["<= 7 Days", "<= 14 Days", "<= 21 Days", "<= 30 Days", "Conversion Range"]

# Initialize results list
results_Z = []


for condition in conditions:
    # Filter data for the current condition
    exposed_conversion = conversion_condition2[(conversion_condition2['purchase_flp_bin'] == condition) & (conversion_condition2['exposure'] != 'Holdout')]['cnt_amp'].values[0]
    holdout_conversion = conversion_condition2[(conversion_condition2['purchase_flp_bin'] == condition) & (conversion_condition2['exposure'] == 'Holdout')]['cnt_amp'].values[0]
    
    # Get base totals
    exposed_total = df_lift_base[df_lift_base['exposure'] != 'Holdout']['cnt_amp'].values[0]
    holdout_total = df_lift_base[df_lift_base['exposure'] == 'Holdout']['cnt_amp'].values[0]
    
    # Calculate conversion rates
    conv_exposed = exposed_conversion / exposed_total * 100
    conv_holdout = holdout_conversion / holdout_total * 100
    
    # Calculate Lift
    lift = ((conv_exposed - conv_holdout) / conv_holdout) * 100 if conv_holdout > 0 else None
    
    # Perform z-test for statistical significance
    z_stat, p_value =  sm.stats.proportions_ztest([exposed_conversion, holdout_conversion], [exposed_total, holdout_total])

    # Append results
    results_Z.append({
        'Campaign Name': df_order_first['campaign_name'].iloc[0],
        'Measurement Method': 'Z-Test (No Prior)',
        'Meansurement Run Date': datetime.today().strftime('%Y-%m-%d'),
        'Campaign Start Date': FLP_date,
        'List Pull End Date': last_listpull_date,  #Max_lp_date,
        #'Conversion Start Date': filter_date.strftime('%Y-%m-%d'),
        'Conversion Start Date': (conv_start_range_date if condition == "Conversion Range" else filter_date).strftime('%Y-%m-%d'),
        'Conversion End Date': conv_end_date_dt.strftime('%Y-%m-%d'),
        'Condition': condition,
        'Conversion Rate - Exposed': f'{conv_exposed:.4f}%',
        'Conversion Rate - Holdout': f'{conv_holdout:.4f}%',
        'Lift (%)': f'{lift:.4f}%',
        'P-Value': f'{p_value:.4f}',
        'Exposed - Conversion': f'{exposed_conversion:,}',
        'Exposed Total': f'{exposed_total:,}',
        'Holdout - Conversion': f'{holdout_conversion:,}',
        'Holdout Total': f'{holdout_total:,}',
    })

# Convert results to DataFrame
results_z_test_df2 = pd.DataFrame(results_Z)

# Display results
#print(results_z_test_df)
print("Results: Z-Test, Remove Prior Purchase")
results_z_test_df2.display()


Results: Z-Test, Remove Prior Purchase


Campaign Name,Measurement Method,Meansurement Run Date,Campaign Start Date,List Pull End Date,Conversion Start Date,Conversion End Date,Condition,Conversion Rate - Exposed,Conversion Rate - Holdout,Lift (%),P-Value,Exposed - Conversion,Exposed Total,Holdout - Conversion,Holdout Total
OnStar_Connected_Paid_Media,Z-Test (No Prior),2025-02-12,2024-10-28,2025-01-17,2024-10-28,2025-02-10,<= 7 Days,0.6379%,0.6390%,-0.1729%,0.9634,"2,746","430,492",935,"146,327"
OnStar_Connected_Paid_Media,Z-Test (No Prior),2025-02-12,2024-10-28,2025-01-17,2024-10-28,2025-02-10,<= 14 Days,1.1080%,1.0948%,1.2081%,0.6758,"4,770","430,492","1,602","146,327"
OnStar_Connected_Paid_Media,Z-Test (No Prior),2025-02-12,2024-10-28,2025-01-17,2024-10-28,2025-02-10,<= 21 Days,1.5080%,1.5342%,-1.7072%,0.4785,"6,492","430,492","2,245","146,327"
OnStar_Connected_Paid_Media,Z-Test (No Prior),2025-02-12,2024-10-28,2025-01-17,2024-10-28,2025-02-10,<= 30 Days,1.8667%,1.8992%,-1.7097%,0.4289,"8,036","430,492","2,779","146,327"
OnStar_Connected_Paid_Media,Z-Test (No Prior),2025-02-12,2024-10-28,2025-01-17,2025-01-01,2025-02-10,Conversion Range,1.6869%,1.7222%,-2.0476%,0.3668,"7,262","430,492","2,520","146,327"


In [0]:
# # Input data
# Conversion = [6238, 1800]  # Conversions: Exposed, Holdout
# Base_Totals = [81587, 26657]  # Totals: Exposed, Holdout

# # Perform the two-proportion z-test
# z_stat, p_value = sm.stats.proportions_ztest(Conversion, Base_Totals)

# print(f"Z-Statistic: {z_stat:.6f}")
# print(f"P-Value: {p_value:.6f}")

In [0]:
# Define Base for lift analysis
df_lift_base = base_total

# Measurement conditions
conditions = ["<= 7 Days", "<= 14 Days", "<= 21 Days", "<= 30 Days","Conversion Range"]

# Initialize results list
results_chi = []

for condition in conditions:
    # Filter data for the current condition
    exposed_conversion = conversion_condition[
        (conversion_condition['purchase_flp_bin'] == condition) & 
        (conversion_condition['exposure'] != 'Holdout')
    ]['cnt_amp'].values[0]
    
    holdout_conversion = conversion_condition[
        (conversion_condition['purchase_flp_bin'] == condition) & 
        (conversion_condition['exposure'] == 'Holdout')
    ]['cnt_amp'].values[0]
    
    # Get base totals
    exposed_total = df_lift_base[df_lift_base['exposure'] != 'Holdout']['cnt_amp'].values[0]
    holdout_total = df_lift_base[df_lift_base['exposure'] == 'Holdout']['cnt_amp'].values[0]
    
    # Calculate non-conversions
    exposed_non_conversion = exposed_total - exposed_conversion
    holdout_non_conversion = holdout_total - holdout_conversion
    
    # Observed data for chi-squared test
    observed = [
        [exposed_conversion, exposed_non_conversion],  # Exposed group
        [holdout_conversion, holdout_non_conversion]   # Holdout group
    ]
    
    # Perform chi-squared test
    chi2_stat, p_value, _, _ = chi2_contingency(observed)
    
    # Calculate conversion rates
    conv_exposed = exposed_conversion / exposed_total * 100
    conv_holdout = holdout_conversion / holdout_total * 100
    
    # Calculate Lift
    lift = ((conv_exposed - conv_holdout) / conv_holdout) * 100 if conv_holdout > 0 else None
    
    # Append results
    results_chi.append({
        'Campaign Name': df_order_first['campaign_name'].iloc[0],
        'Measurement Method': 'Chi-Square-Test',
        'Meansurement Run Date': datetime.today().strftime('%Y-%m-%d'),
        'Campaign Start Date': FLP_date,
        'List Pull End Date': last_listpull_date,  #Max_lp_date,
        #'Conversion Start Date': filter_date.strftime('%Y-%m-%d'),
        'Conversion Start Date': (conv_start_range_date if condition == "Conversion Range" else filter_date).strftime('%Y-%m-%d'),
        'Conversion End Date': conv_end_date_dt.strftime('%Y-%m-%d'),
        'Condition': condition,
        'Conversion Rate - Exposed': f'{conv_exposed:.4f}%',
        'Conversion Rate - Holdout': f'{conv_holdout:.4f}%',
        'Lift (%)': f'{lift:.4f}%',
        'P-Value': f'{p_value:.4f}',
        'Chi-Squared Stat': f'{chi2_stat:.4f}',
        'Exposed - Conversion': f'{exposed_conversion:,}',
        'Exposed Total': f'{exposed_total:,}',
        'Holdout - Conversion': f'{holdout_conversion:,}',
        'Holdout Total': f'{holdout_total:,}',
    })

# Convert results to DataFrame
results_chi_test_df = pd.DataFrame(results_chi)

# Display results
results_chi_test_df.display()


Campaign Name,Measurement Method,Meansurement Run Date,Campaign Start Date,List Pull End Date,Conversion Start Date,Conversion End Date,Condition,Conversion Rate - Exposed,Conversion Rate - Holdout,Lift (%),P-Value,Chi-Squared Stat,Exposed - Conversion,Exposed Total,Holdout - Conversion,Holdout Total
OnStar_Connected_Paid_Media,Chi-Square-Test,2025-02-12,2024-10-28,2025-01-17,2024-10-28,2025-02-10,<= 7 Days,0.6586%,0.6642%,-0.8518%,0.8305,0.0458,"2,875","436,543",986,"148,440"
OnStar_Connected_Paid_Media,Chi-Square-Test,2025-02-12,2024-10-28,2025-01-17,2024-10-28,2025-02-10,<= 14 Days,1.1389%,1.1385%,0.0388%,1.0000,0.0000,"4,972","436,543","1,690","148,440"
OnStar_Connected_Paid_Media,Chi-Square-Test,2025-02-12,2024-10-28,2025-01-17,2024-10-28,2025-02-10,<= 21 Days,1.5483%,1.5872%,-2.4491%,0.3018,1.0660,"6,759","436,543","2,356","148,440"
OnStar_Connected_Paid_Media,Chi-Square-Test,2025-02-12,2024-10-28,2025-01-17,2024-10-28,2025-02-10,<= 30 Days,1.9137%,1.9604%,-2.3830%,0.2625,1.2555,"8,354","436,543","2,910","148,440"
OnStar_Connected_Paid_Media,Chi-Square-Test,2025-02-12,2024-10-28,2025-01-17,2025-01-01,2025-02-10,Conversion Range,1.6635%,1.6977%,-2.0105%,0.3820,0.7642,"7,262","436,543","2,520","148,440"


In [0]:
# Define Base for lift analysis
df_lift_base = base_total_no_prior

# Measurement conditions
conditions = ["<= 7 Days", "<= 14 Days", "<= 21 Days", "<= 30 Days","Conversion Range"]

# Initialize results list
results_chi = []

for condition in conditions:
    # Filter data for the current condition
    exposed_conversion = conversion_condition2[
        (conversion_condition2['purchase_flp_bin'] == condition) & 
        (conversion_condition2['exposure'] != 'Holdout')
    ]['cnt_amp'].values[0]
    
    holdout_conversion = conversion_condition2[
        (conversion_condition2['purchase_flp_bin'] == condition) & 
        (conversion_condition2['exposure'] == 'Holdout')
    ]['cnt_amp'].values[0]
    
    # Get base totals
    exposed_total = df_lift_base[df_lift_base['exposure'] != 'Holdout']['cnt_amp'].values[0]
    holdout_total = df_lift_base[df_lift_base['exposure'] == 'Holdout']['cnt_amp'].values[0]
    
    # Calculate non-conversions
    exposed_non_conversion = exposed_total - exposed_conversion
    holdout_non_conversion = holdout_total - holdout_conversion
    
    # Observed data for chi-squared test
    observed = [
        [exposed_conversion, exposed_non_conversion],  # Exposed group
        [holdout_conversion, holdout_non_conversion]   # Holdout group
    ]
    
    # Perform chi-squared test
    chi2_stat, p_value, _, _ = chi2_contingency(observed)
    
    # Calculate conversion rates
    conv_exposed = exposed_conversion / exposed_total * 100
    conv_holdout = holdout_conversion / holdout_total * 100
    
    # Calculate Lift
    lift = ((conv_exposed - conv_holdout) / conv_holdout) * 100 if conv_holdout > 0 else None
    
    # Append results
    results_chi.append({
        'Campaign Name': df_order_first['campaign_name'].iloc[0],
        'Measurement Method': 'Chi-Square-Test (No Prior)',
        'Meansurement Run Date': datetime.today().strftime('%Y-%m-%d'),
        'Campaign Start Date': FLP_date,
        'List Pull End Date': last_listpull_date,  #Max_lp_date,
        #'Conversion Start Date': filter_date.strftime('%Y-%m-%d'),
        'Conversion Start Date': (conv_start_range_date if condition == "Conversion Range" else filter_date).strftime('%Y-%m-%d'),
        'Conversion End Date': conv_end_date_dt.strftime('%Y-%m-%d'),
        'Condition': condition,
        'Conversion Rate - Exposed': f'{conv_exposed:.4f}%',
        'Conversion Rate - Holdout': f'{conv_holdout:.4f}%',
        'Lift (%)': f'{lift:.4f}%',
        'P-Value': f'{p_value:.4f}',
        'Chi-Squared Stat': f'{chi2_stat:.4f}',
        'Exposed - Conversion': f'{exposed_conversion:,}',
        'Exposed Total': f'{exposed_total:,}',
        'Holdout - Conversion': f'{holdout_conversion:,}',
        'Holdout Total': f'{holdout_total:,}',
    })

# Convert results to DataFrame
results_chi_test_df = pd.DataFrame(results_chi)

# Display results
print("Results: Chi-Square, Remove Prior Purchase")
results_chi_test_df.display()


Results: Chi-Square, Remove Prior Purchase


Campaign Name,Measurement Method,Meansurement Run Date,Campaign Start Date,List Pull End Date,Conversion Start Date,Conversion End Date,Condition,Conversion Rate - Exposed,Conversion Rate - Holdout,Lift (%),P-Value,Chi-Squared Stat,Exposed - Conversion,Exposed Total,Holdout - Conversion,Holdout Total
OnStar_Connected_Paid_Media,Chi-Square-Test (No Prior),2025-02-12,2024-10-28,2025-01-17,2024-10-28,2025-02-10,<= 7 Days,0.6379%,0.6390%,-0.1729%,0.9786,0.0007,"2,746","430,492",935,"146,327"
OnStar_Connected_Paid_Media,Chi-Square-Test (No Prior),2025-02-12,2024-10-28,2025-01-17,2024-10-28,2025-02-10,<= 14 Days,1.1080%,1.0948%,1.2081%,0.6864,0.1630,"4,770","430,492","1,602","146,327"
OnStar_Connected_Paid_Media,Chi-Square-Test (No Prior),2025-02-12,2024-10-28,2025-01-17,2024-10-28,2025-02-10,<= 21 Days,1.5080%,1.5342%,-1.7072%,0.4862,0.4849,"6,492","430,492","2,245","146,327"
OnStar_Connected_Paid_Media,Chi-Square-Test (No Prior),2025-02-12,2024-10-28,2025-01-17,2024-10-28,2025-02-10,<= 30 Days,1.8667%,1.8992%,-1.7097%,0.4354,0.6083,"8,036","430,492","2,779","146,327"
OnStar_Connected_Paid_Media,Chi-Square-Test (No Prior),2025-02-12,2024-10-28,2025-01-17,2025-01-01,2025-02-10,Conversion Range,1.6869%,1.7222%,-2.0476%,0.3730,0.7936,"7,262","430,492","2,520","146,327"


In [0]:
# # Input data (successes and failures for both groups)
# data = [
#     [6809, 3720091 - 6809],  # Exposed: Successes, Failures
#     [2022, 1249404 - 2022]   # Holdout: Successes, Failures
# ]

# # Perform chi-squared test
# chi2, p_value, _, _ = chi2_contingency(data)

# print(f"Chi-Squared Statistic: {chi2:.6f}")
# print(f"P-Value: {p_value:.6f}")


# 7. Ad-hoc Measurement

In [0]:
%sql
--select * from amp_order_first_v limit 1;
--select * from amp_order_hist_v limit 1;

--select distinct latest_purchase_group  from amp_order_hist_v;

In [0]:
%sql

DROP VIEW IF EXISTS amp_order_v;
CREATE temp view amp_order_v as (

with temp_base as (  
  SELECT aud.*,
    row_nm_order,
    CREATE_MKTG_CHANNL_CD,
    prod_id,
    PRICE_PLAN_CD, 
    PROD_TYPE_CD,
    to_date(create_timstm) as purchase_date,
    datediff(purchase_date, first_lp_date) as purchase_flp,
    
    -- Define prior_purchase if purchased happend prior FLP for each
    CASE WHEN purchase_date < first_lp_date and PRICE_PLAN_CD in ('COMP') then -1 
      WHEN purchase_date < first_lp_date and PRICE_PLAN_CD <> ('COMP') then 1 
      WHEN  purchase_date >= first_lp_date then 0 
      WHEN purchase_date is null then NULL
      else 999 END as prior_purchase,

    ord_h.row_nm_last_odr,
    ord_h.latest_purchase_group


  FROM Campaign_dmp_v as aud 
    LEFT JOIN order_gaa_v as ord 
    ON aud.vin = ord.vin 
    AND aud.account_number = ord.account_number

    LEFT JOIN 
      (select distinct amperity_id, account_number, vin,row_nm_last_odr, latest_purchase_group 
        from amp_order_hist_v where row_nm_last_odr = 1 or row_nm_last_odr is null
      ) as ord_h
      on aud.amperity_id = ord_h.amperity_id
      AND aud.account_number = ord_h.account_number
      AND aud.vin = ord_h.vin

  WHERE Acct_VIN_Status not in ("No VIN") -- Remove conditions: Missing VIN
  --WHERE Acct_VIN_Status not in ("No VIN", "Neither")
  )

select a.*, 
  max_prior_purchase,
  min_prior_purchase,
  
  CASE WHEN prior_purchase = -1 and max_prior_purchase = -1 THEN 0 -- Prior purchased COMP and no future Pruchased
    WHEN min_prior_purchase = -1 and max_prior_purchase = 1 THEN 0 -- Prior purchased COMP and Other Packages | USE MIN & MAX
    ELSE max_prior_purchase
    END as prior_purchase_status,

  --Row Number subscriptions based on prior_purchase status, will count first purchase for prior and post status
  row_number() OVER (PARTITION BY a.amperity_id, a.account_number, a.vin, a.prior_purchase order by purchase_date) as row_nm_order_prior,
  --Row Number subscriptions based on purchase_date, count 1 purchase per day, all purchases will be counted
  row_number() OVER (PARTITION BY a.amperity_id, a.account_number, a.vin, purchase_date order by prod_id) as row_nm_order_per_day

  FROM temp_base as a
  LEFT JOIN 
    (select amperity_id, account_number, vin, audience_nm, 
      max(prior_purchase) as max_prior_purchase,
      min(prior_purchase) as min_prior_purchase
    FROM temp_base group by ALL
    ) as b 
  ON a.amperity_id = b.amperity_id 
  and a.account_number = b.account_number 
  and a.vin = b.vin 
  and a.audience_nm = b.audience_nm 

  WHERE Acct_VIN_Status not in ("No VIN") -- Remove conditions: Missing VIN
  --WHERE Acct_VIN_Status not in ("No VIN", "Neither")
);




--- Filter to ever first order for prior and post purchase status
DROP VIEW IF EXISTS amp_order_first_v;
create temp view amp_order_first_v as (
  SELECT * FROM amp_order_v 
    where row_nm_order_prior = 1
);


--- Filter to all order, only 1st order per day
DROP VIEW IF EXISTS amp_order_all_v;
create temp view amp_order_all_v as (
  SELECT * FROM amp_order_v 
    where row_nm_order_per_day = 1
);





In [0]:
%sql
-- select purchase_date, first_lp_date, purchase_flp, 
--   campaign_name, 
--   exposure, 
--   prior_purchase_status,
--   case when purchase_date is not null then 1 else 0 END as purchase,
--   count(distinct amperity_id) as cnt_amp,
--   latest_purchase_group
-- from amp_order_first_v
-- group by all
-- order by purchase_date

In [0]:
%sql
-- select * from amp_order_first_v where latest_purchase_group is null 
-- order by amperity_id, account_number, VIN limit 100;

In [0]:
%sql
--select * from amp_order_hist_v where amperity_id in ("000031c3-1c4c-3919-a505-a2d32f0603e3")

In [0]:
%sql
--select * from amp_order_v  where amperity_id in ("000031c3-1c4c-3919-a505-a2d32f0603e3") 